<a href="https://colab.research.google.com/github/allarom/advanced-genai-26/blob/dongy/multi-agent-step-2_strategy-A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Agent Orchestration for RAG Systems
# Step 2. Specialized Agents for Orchestrated RAG

*Assignee: Alla* - *Review:*

This notebook implements the specialized-agent layer for a modular RAG system. It defines role-based agents with a shared `AgentState` contract and keeps retriever artifacts reusable for later orchestration experiments.

The design is intentionally decoupled from routing, so external orchestrators can apply strategies such as Parallel+Fusion, Sequential Waterfall, or Confidence-Based Routing without changing agent internals.



In [6]:
# Colab setup
%pip install -q langchain-core langchain-community langchain-huggingface chromadb \
  rank-bm25 langdetect nltk sentence-transformers
# Install missing packages (not in baseline requirements.txt)
!pip install -q matplotlib scipy


In [7]:
import os
import re
import json
import pickle
import random
import pathlib
from dataclasses import dataclass
from collections import defaultdict
from typing import Any

import numpy as np
import nltk
from langdetect import detect

from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words('english'))
STOP_DE = set(nltk.corpus.stopwords.words('german'))

print('Setup complete.')



Setup complete.


In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Setup

Requirements:
- Add `HF_TOKEN` to Colab Secrets (https://huggingface.co/settings/tokens).
- Ensure Google Drive contains:
  - `/content/drive/MyDrive/Adv_GenAI/benchmark`
  - `/content/drive/MyDrive/Adv_GenAI/storage`

Scope selection:
- `EVAL_SCOPE = 'full_corpus'` or `EVAL_SCOPE = 'subsample'`
- For orchestration comparison, `full_corpus` is recommended.



In [9]:
# Paths (edit PROJECT_ROOT if needed)
# PROJECT_ROOT must be the folder that contains benchmark/ and storage/
CANDIDATE_ROOTS = [
    pathlib.Path('/content/drive/MyDrive/Adv_GenAI'),
    pathlib.Path('/content/drive/MyDrive/advanced-genai-26/baseline/advanced_genAI-main/data'),
    pathlib.Path('/content/drive/MyDrive/advanced_genAI-main/data'),
]

def looks_like_project_root(p: pathlib.Path) -> bool:
    return (p / 'benchmark').exists() and (p / 'storage').exists()

PROJECT_ROOT = None
for c in CANDIDATE_ROOTS:
    if looks_like_project_root(c):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not auto-detect project root. Set PROJECT_ROOT manually.')

PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)

# Choose scope: 'subsample' or 'full_corpus'
EVAL_SCOPE = 'full_corpus'
assert EVAL_SCOPE in {'subsample', 'full_corpus'}
print('EVAL_SCOPE =', EVAL_SCOPE)

if EVAL_SCOPE == 'subsample':
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl'
    if not PATH_BM25_PICKLE.exists():
        PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval/fixed_size_chunk/bm25_retriever.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/subsample/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/subsample/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/subsample/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'
else:
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/full_corpus/retrieval/fixed_size_chunk/bm25_retriever_full.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/full_corpus/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/full_corpus/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/full_corpus/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'

required = [PATH_BM25_PICKLE, PATH_DENSE_INDEX, PATH_GRAG_ROOT, PATH_CHUNK_PKL]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('All required retrieval artifacts found.')



PROJECT_ROOT = /content/drive/MyDrive/Adv_GenAI
EVAL_SCOPE = full_corpus
All required retrieval artifacts found.


## 2. Multi-Agent Architecture (Specialized Roles)

This is the main implementation section.

Implemented roles:
- Query Understanding Agent
- Retriever Agents (`BM25`, `Dense`, `GraphRAG`)
- Fusion Agent (merge + deduplicate)
- Re-Ranker Agent
- Answer Synthesizer Agent
- Critic Agent (grounding check + re-retrieval trigger)

All agents read/write a shared `AgentState`, making routing strategies interchangeable.



## 2.1 Retrieval Component: BM25 Adapter

Loads BM25 artifacts with a compatibility wrapper and exposes a stable `search(query, top_k)` interface across artifact variants.

**Retriever Compatibility Note**
Some BM25 artifacts were serialized with custom classes (e.g., `BilingualBM25` or `QEBM25`). During `pickle.load(...)`, Python must resolve those class definitions to reconstruct the object, even if we later access the retriever only through `BM25RetrieverAdapter`.

For this reason, these class definitions are intentionally kept in the notebook as deserialization shims for cross-version artifact compatibility.




In [10]:
# Robust BM25 loader for both subsample and full-corpus pickle formats
class BilingualBM25:
    """Compatibility class for notebook pickles."""

    def _rank_lang(self, q: str, lang: str, k: int):
        # Subsample-style object: self.bm25 + self.docs_by_lang
        try:
            q_tokens = nltk.word_tokenize(q)
        except Exception:
            q_tokens = q.split()
        scores = self.bm25[lang].get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata['bm25_score'] = float(scores[i])
            hits.append(d)
        return hits

    def _get_docs_with_scores(self, ret, qq, top_k):
        # Full-corpus-style object: self.retrievers
        if hasattr(ret, 'get_relevant_documents_with_scores'):
            try:
                return ret.get_relevant_documents_with_scores(qq, k=top_k)
            except Exception:
                pass

        if hasattr(ret, 'vectorizer') and hasattr(ret, 'docs'):
            try:
                toks = qq.lower().split()
                scores = ret.vectorizer.get_scores(toks)
                ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
                return [(ret.docs[idx], float(score)) for idx, score in ranked]
            except Exception:
                pass

        if hasattr(ret, 'invoke'):
            try:
                old_k = getattr(ret, 'k', None)
                if old_k is not None:
                    ret.k = top_k
                docs = ret.invoke(qq)
                if old_k is not None:
                    ret.k = old_k
                return [(d, d.metadata.get('score', 0.0)) for d in docs[:top_k]]
            except Exception:
                pass

        return []

    def search(self, query: str, top_k: int = 100):
        # Route to correct behavior based on available attributes
        if hasattr(self, 'bm25') and hasattr(self, 'docs_by_lang'):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)
            for lang in ('en', 'de'):
                q_lang = translator.translate(query, lang) if translator and lang != src else query
                bag.extend(self._rank_lang(q_lang, lang, top_k))

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid not in best or d.metadata['bm25_score'] > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d
            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        if hasattr(self, 'retrievers') and isinstance(self.retrievers, dict):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)

            for lang, ret in self.retrievers.items():
                qq = translator.translate(query, lang) if translator and lang != src else query
                docs_with_scores = self._get_docs_with_scores(ret, qq, top_k)
                for doc, score in docs_with_scores:
                    doc.metadata['bm25_score'] = float(score)
                    bag.append(doc)

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid is None:
                    continue
                if uid not in best or d.metadata.get('bm25_score', -1e9) > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d

            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        raise AttributeError('Unsupported BilingualBM25 object format.')

class QEBM25:
    @staticmethod
    def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
        def tok(text: str):
            try:
                return nltk.word_tokenize(text.lower())
            except Exception:
                return text.lower().split()

        hits = base_retriever.search(query, top_k=fb_docs)
        tokens = [
            t for h in hits for t in tok(h.page_content)
            if t.isalpha() and t not in STOP_EN and t not in STOP_DE
        ]
        extra = ' '.join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
        return f'{query} {extra}' if extra else query

    def search(self, query: str, top_k: int = 100):
        if hasattr(self, 'base'):
            expanded = self._expand_query(query, self.base)
            return self.base.search(expanded, top_k)
        raise AttributeError('QEBM25 object missing base retriever.')

with open(PATH_BM25_PICKLE, 'rb') as f:
    bm25_raw = pickle.load(f)

class BM25RetrieverAdapter:
    def __init__(self, obj):
        self.obj = obj

    def search(self, query: str, top_k: int = 100):
        # Primary path
        if hasattr(self.obj, 'search'):
            try:
                return self.obj.search(query, top_k=top_k)
            except TypeError:
                return self.obj.search(query, k=top_k)

        # LangChain retriever fallback
        if hasattr(self.obj, 'invoke'):
            old_k = getattr(self.obj, 'k', None)
            if old_k is not None:
                self.obj.k = top_k
            docs = self.obj.invoke(query)
            if old_k is not None:
                self.obj.k = old_k
            for rank, d in enumerate(docs, start=1):
                if hasattr(d, 'metadata'):
                    d.metadata.setdefault('bm25_score', float(top_k - rank))
            return docs[:top_k]

        raise AttributeError(f'Unsupported BM25 object type: {type(self.obj)}')

bm25_retriever = BM25RetrieverAdapter(bm25_raw)
print('BM25 loaded:', type(bm25_raw), '-> adapter ready')


BM25 loaded: <class '__main__.BilingualBM25'> -> adapter ready


### Explanation: BM25 compatibility loader + adapter

This cell handles different BM25 artifact formats from previous runs.

What it ensures:
1. Old pickles can be deserialized safely (`BilingualBM25`, `QEBM25`).
2. Different object styles still expose one interface: `search(query, top_k)`.
3. Result docs carry comparable metadata (e.g., `bm25_score`).

`BM25RetrieverAdapter` is the final stable wrapper used by the pipeline.

## 2.2 Retrieval Component: Dense Retriever

Initializes the multilingual E5 + Chroma retriever and returns dense candidates through the same interface used by other retrievers.



In [11]:
# Dense retriever
class DenseRetriever:
    def __init__(self, index_dir: pathlib.Path, model_name='intfloat/multilingual-e5-large-instruct', k: int = 100):
        self.k = k
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda' if os.path.exists('/proc/driver/nvidia/version') else 'cpu'},
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True},
        )
        self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)

    def _prep(self, q: str) -> str:
        return 'query: ' + q.strip()

    def search(self, query: str, top_k: int = 100):
        k = top_k or self.k
        hits = self.store.similarity_search_with_score(self._prep(query), k=k)
        out = []
        for doc, dist in hits:
            doc.metadata['dense_score'] = 1.0 - float(dist)
            out.append(doc)
        return out

dense_retriever = DenseRetriever(PATH_DENSE_INDEX, k=100)
print('Dense retriever ready.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/tmp/ipykernel_3033/1020676855.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)


Dense retriever ready.


### Explanation: `DenseRetriever`

Dense retrieval uses multilingual embeddings and vector search.

How it works:
1. Prefix query as `query: ...` for E5-style embedding format.
2. Run Chroma similarity search.
3. Convert distances to similarity-style score (`dense_score = 1 - dist`).
4. Return ranked semantic candidates.

This helps with paraphrases and semantic meaning beyond exact keyword match.

## 2.3 Retrieval Component: GraphRAG Retriever

Loads GraphRAG resources and returns graph-informed candidates by selecting relevant communities and ranking chunks within them.



In [12]:
# GraphRAG retriever
class GraphRAGRetriever:
    def __init__(self, graph_root: pathlib.Path, chunk_pkl: pathlib.Path):
        self.root = graph_root
        self.emb_dir = graph_root / 'embeddings'
        self.chunk_pkl = chunk_pkl
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self._emb_cache = {}
        self._cid_cache = {}
        self._chunk_by_id = None
        self._chunk_vec_cache = {}
        self.comm2chunk = json.loads((self.root / 'comm2chunk_fixed.json').read_text(encoding='utf-8'))

        # Auto-detect which community levels are available (C0, C1, C2, ...)
        # so the notebook works regardless of which level the team generated.
        self._available_levels = []
        if self.emb_dir.exists():
            for p in sorted(self.emb_dir.glob('EMB_fixed_C*.npy')):
                try:
                    level_num = int(p.stem.split('C')[-1])
                    self._available_levels.append(level_num)
                except ValueError:
                    continue
        self._default_level = self._available_levels[0] if self._available_levels else 1
        print(f'[GraphRAG] Available community levels: C{self._available_levels}')
        print(f'[GraphRAG] Using default level C{self._default_level}')

    def _load_embeddings(self, level: int):
        if level in self._emb_cache:
            return self._emb_cache[level], self._cid_cache[level]
        mat = np.load(self.emb_dir / f'EMB_fixed_C{level}.npy')
        cid = json.loads((self.emb_dir / f'CID_fixed_C{level}.json').read_text(encoding='utf-8'))
        self._emb_cache[level] = mat
        self._cid_cache[level] = cid
        return mat, cid

    def _load_chunks(self):
        if self._chunk_by_id is not None:
            return self._chunk_by_id
        with open(self.chunk_pkl, 'rb') as f:
            docs_norm = pickle.load(f)

        def restore(d):
            raw = d.metadata.get('original_text') or d.page_content
            return Document(page_content=raw, metadata=d.metadata)
        docs = [restore(d) for d in docs_norm]
        self._chunk_by_id = {d.metadata['chunk_id']: d for d in docs}
        return self._chunk_by_id

    def _chunk_vec(self, cid: str, chunks: dict):
        if cid not in self._chunk_vec_cache:
            self._chunk_vec_cache[cid] = self.embedder.encode([chunks[cid].page_content], normalize_embeddings=True)[0]
        return self._chunk_vec_cache[cid]

    def retrieve(self, query: str, level: str = None, k_comms: int = 24, top_k: int = 100):
        # Default to auto-detected level (e.g. C2 if that is what the team generated)
        if level is None:
            L = self._default_level
        else:
            L = int(level.lstrip('C'))
        emb_mat, cid_list = self._load_embeddings(L)
        chunks = self._load_chunks()

        q_vec = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims_comm = emb_mat @ q_vec
        best_idx = sims_comm.argsort()[::-1][:k_comms]

        cand_ids = set()
        for idx in best_idx:
            cand_ids.update(self.comm2chunk.get(cid_list[idx], []))

        scored = []
        for cid in cand_ids:
            if cid not in chunks:
                continue
            sim = float(self._chunk_vec(cid, chunks) @ q_vec)
            scored.append((cid, sim))

        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:top_k]

        out = []
        for cid, sim in scored:
            d = chunks[cid]
            d.metadata['grag_score'] = (sim + 1.0) / 2.0
            out.append(d)
        return out

    def search(self, query: str, top_k: int = 100, k_comms: int = 48):
        # search() is the agent-facing entry point.
        # It uses the auto-detected default level instead of hardcoded C1.
        return self.retrieve(query=query, level=None, k_comms=k_comms, top_k=top_k)


graph_retriever = GraphRAGRetriever(PATH_GRAG_ROOT, PATH_CHUNK_PKL)
print('GraphRAG retriever ready.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[GraphRAG] Available community levels: C[0, 1, 2]
[GraphRAG] Using default level C0
GraphRAG retriever ready.


### Explanation: `GraphRAGRetriever`

This retriever is graph-guided + semantic.

Step-by-step:
1. Load community embeddings and chunk mappings.
2. Encode query embedding.
3. Select top communities (`k_comms`).
4. Gather chunk candidates from those communities.
5. Score candidates by embedding similarity.
6. Return top `top_k` docs with `grag_score` metadata.

Good for relationship/context-heavy questions.

In [13]:
# Shared utilities used by Step 2 agents
from collections import defaultdict
from typing import Any

def _uid(doc: Any):
    meta = getattr(doc, 'metadata', {}) or {}
    return meta.get('chunk_id') or meta.get('record_id') or meta.get('doc_id')

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights=None):
    # Balanced defaults: Dense leads (better semantic coverage),
    # BM25 equal (strong for factoid), Graph gets a fair share.
    weights = weights or {'bm25': 1.0, 'dense': 1.2, 'graph': 0.8}
    scores = defaultdict(float)
    store = {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata['fused_score'] = float(scores[_uid(d)])
    return fused

def _token_set(text: str):
    text = (text or '').lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return set(t for t in text.split() if t)

def _overlap_rerank(docs, query: str, top_k: int):
    q_terms = set(t.lower() for t in query.split() if t.strip())
    scored = []
    for d in docs:
        text = (d.metadata.get('original_text') or d.page_content or '').lower()
        overlap = len(q_terms & set(text.split())) / max(len(q_terms), 1)
        scored.append((overlap, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

print('Shared Step 2 utilities ready.')


Shared Step 2 utilities ready.


### Explanation: Shared utilities

These helpers are used by multiple agents:
- `_uid(...)`: extract stable document ID.
- `_safe_unique(...)`: remove duplicates / invalid IDs.
- `_rrf_fuse(...)`: weighted reciprocal-rank fusion.
- `_token_set(...)`: normalization + tokenization.
- `_overlap_rerank(...)`: simple fallback reranker.

They keep behavior consistent across pipeline stages.

In [14]:
# Multi-agent role contracts
from dataclasses import dataclass, field
from typing import Dict, List, Any

@dataclass
class AgentState:
    query: str
    normalized_query: str = ''
    query_type: str = 'mixed'
    query_hints: Dict[str, float] = field(default_factory=dict)
    retrieval_by_agent: Dict[str, List[Any]] = field(default_factory=dict)
    fused_docs: List[Any] = field(default_factory=list)
    reranked_docs: List[Any] = field(default_factory=list)
    final_answer: str = ''
    evidence_ids: List[str] = field(default_factory=list)
    critic_ok: bool = False
    critic_feedback: str = ''
    needs_reretrieval: bool = False

class BaseAgent:
    name = 'base'
    def run(self, state: AgentState, **kwargs) -> AgentState:
        raise NotImplementedError

print('Agent contracts ready.')



Agent contracts ready.


### Explanation: `AgentState` and base contract

`AgentState` is the shared memory object passed through all agents.

Main fields:
- `query`, `normalized_query`, `query_type`
- `query_hints` (weights + hints)
- `retrieval_by_agent`, `fused_docs`, `reranked_docs`
- `final_answer`, `evidence_ids`
- `critic_ok`, `critic_feedback`, `needs_reretrieval`

`BaseAgent.run(...)` gives one consistent interface for all agent components.

In [15]:
# 1) Query Understanding Agent
import re as _re

class QueryUnderstandingAgent(BaseAgent):
    name = 'query_understanding'

    # Matches "who was/is/were [role]" or "who served as" patterns
    _WHO_ROLE_RE = _re.compile(r'^who\b', _re.I)  # any question starting with 'who'
    # Matches a 4-digit year anywhere in the query
    _YEAR_RE = _re.compile(r'\b(1\d{3}|20\d{2})\b')

    def run(self, state: AgentState, **kwargs) -> AgentState:
        q = (state.query or '').strip()
        q_low = q.lower()

        state.normalized_query = ' '.join(q.split())

        graph_signals = {'relationship', 'connected', 'connection', 'connections', 'dependency', 'impact', 'between'}
        keyword_signals = {'exactly', 'define', 'list', 'when', 'where', 'who'}

        gr_score  = sum(1 for t in graph_signals  if t in q_low)
        kw_score  = sum(1 for t in keyword_signals if t in q_low)

        has_year        = bool(self._YEAR_RE.search(q))
        who_role_query  = bool(self._WHO_ROLE_RE.match(q))
        factual_starts  = (
            'when ', 'where ', 'which ',
            'what year', 'what date',
            'what is ', 'what are ', 'what was ', 'what were ',
            'what does ', 'what do ', 'what did ',
        )
        factual_query   = q_low.startswith(factual_starts)

        # "Who was/is [role]" + year → entity-temporal: graph community search
        # is best for finding person-role-time triples.
        if who_role_query and has_year:
            q_type = 'entity_temporal'
        elif who_role_query:
            q_type = 'entity'
        elif gr_score >= 1:
            q_type = 'graph'
        elif factual_query or kw_score >= 1:
            q_type = 'keyword'
        elif len(q.split()) >= 7:
            q_type = 'semantic'
        else:
            q_type = 'mixed'

        state.query_type = q_type

        # Store year in state so downstream agents can use it
        m = self._YEAR_RE.search(q)
        state.query_hints['_year'] = int(m.group()) if m else None

        # Fusion weights by type
        if q_type == 'entity_temporal':
            # Graph captures person-role-org community structure best;
            # Dense for semantic similarity; BM25 suppressed (year matches noise).
            state.query_hints.update({'bm25': 0.7, 'dense': 1.1, 'graph': 1.4})
        elif q_type == 'entity':
            state.query_hints.update({'bm25': 0.9, 'dense': 1.1, 'graph': 1.3})
        elif q_type == 'keyword':
            state.query_hints.update({'bm25': 1.3, 'dense': 1.1, 'graph': 0.7})
        elif q_type == 'semantic':
            state.query_hints.update({'bm25': 0.8, 'dense': 1.4, 'graph': 0.9})
        elif q_type == 'graph':
            state.query_hints.update({'bm25': 0.8, 'dense': 1.0, 'graph': 1.4})
        else:  # mixed
            state.query_hints.update({'bm25': 1.0, 'dense': 1.2, 'graph': 0.8})

        return state

print('Query Understanding Agent ready.')


Query Understanding Agent ready.


### Explanation: `QueryUnderstandingAgent`

Purpose: interpret query type and set retrieval weights.

Key actions:
1. Normalize query text.
2. Detect signals (factoid/semantic/graph/entity-temporal).
3. Assign `state.query_type`.
4. Store year hint in `state.query_hints['_year']` when present.
5. Set dynamic retriever weights (`bm25`, `dense`, `graph`).

Why useful: different query types benefit from different retrieval emphasis.

In [16]:
# 2) Retriever Agents (BM25, Dense, GraphRAG)
class BM25RetrieverAgent(BaseAgent):
    name = 'bm25_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['bm25'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

class DenseRetrieverAgent(BaseAgent):
    name = 'dense_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['dense'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

class GraphRetrieverAgent(BaseAgent):
    name = 'graph_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['graph'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k, k_comms=48))
        return state

print('Retriever Agents ready.')



Retriever Agents ready.


### Explanation: Retriever Agents (`BM25`, `Dense`, `Graph`)

These wrappers standardize retrieval calls.

What each `run(...)` does:
1. Read `state.normalized_query`.
2. Call retriever with `top_k`.
3. Clean results via `_safe_unique`.
4. Store per-agent outputs in `state.retrieval_by_agent`.

This makes downstream fusion independent of retriever internals.

In [17]:
# 3) Fusion Agent
class FusionAgent(BaseAgent):
    name = 'fusion'

    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        runs = {
            'bm25': state.retrieval_by_agent.get('bm25', []),
            'dense': state.retrieval_by_agent.get('dense', []),
            'graph': state.retrieval_by_agent.get('graph', []),
        }
        fused = _rrf_fuse(runs, weights=state.query_hints or None)
        state.fused_docs = _safe_unique(fused)[:top_k]
        return state

print('Fusion Agent ready.')



Fusion Agent ready.


### Explanation: `FusionAgent`

Purpose: merge outputs from BM25, Dense, and Graph retrievers into one ranked list.

How fusion works:
1. Collect runs from `state.retrieval_by_agent`.
2. Apply weighted RRF (`_rrf_fuse`) using `state.query_hints` when available.
3. Remove duplicates/invalid IDs (`_safe_unique`).
4. Keep top fused documents in `state.fused_docs`.

RRF merges ranked outputs by document IDs (not by merging text chunks).

In [18]:
# 4) Re-Ranker Agent — CrossEncoder with overlap fallback
class ReRankerAgent(BaseAgent):
    name = 'reranker'
    _ce_model = None  # class-level cache; loaded once per session

    @classmethod
    def _get_cross_encoder(cls):
        if cls._ce_model is None:
            print('[ReRanker] Loading CrossEncoder (ms-marco-MiniLM-L-6-v2)...')
            cls._ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            print('[ReRanker] CrossEncoder ready.')
        return cls._ce_model

    def run(self, state: AgentState, top_k: int = 10, **kwargs) -> AgentState:
        docs = state.fused_docs
        if not docs:
            state.reranked_docs = []
            return state

        query = state.normalized_query
        # Limit cross-encoder to top-50 fused candidates to keep latency reasonable
        candidates = docs[:50]

        try:
            model = self._get_cross_encoder()
            pairs = [
                (query, (d.metadata.get('original_text') or d.page_content or '').strip()[:512])
                for d in candidates
            ]
            scores = model.predict(pairs)
            ranked = sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)
            for d, score in ranked:
                d.metadata['rerank_score'] = float(score)
            state.reranked_docs = [d for d, _ in ranked[:top_k]]
        except Exception as e:
            # Overlap rerank as fallback so the pipeline never silently breaks
            print(f'[ReRanker] CrossEncoder failed ({e}), using overlap fallback.')
            state.reranked_docs = _overlap_rerank(docs, query, top_k=top_k)
            for d in state.reranked_docs:
                d.metadata.setdefault('rerank_score', 0.0)

        return state

print('Re-Ranker Agent ready (CrossEncoder + overlap fallback).')


Re-Ranker Agent ready (CrossEncoder + overlap fallback).


### Explanation: `ReRankerAgent`

Purpose: improve ranking quality of top fused candidates.

Step-by-step:
1. Start from `state.fused_docs`.
2. Limit candidates to top 50 for speed.
3. Build `(query, doc_text)` pairs.
4. CrossEncoder predicts relevance scores.
5. Sort by score descending.
6. Keep top `top_k` in `state.reranked_docs`.
7. If CrossEncoder fails, fallback to overlap rerank.

It reranks document order; it does not change document content.

In [19]:
# 5) Answer Synthesizer Agent — temporal-aware extractive scoring
class AnswerSynthesizerAgent(BaseAgent):
    name = 'answer_synthesizer'

    def _build_context(self, docs: List[Any], max_docs: int = 5) -> str:
        parts = []
        for d in docs[:max_docs]:
            txt = (d.metadata.get('original_text') or d.page_content or '').strip()
            if txt:
                parts.append(txt)
        return "\n\n".join(parts)

    def run(self, state: AgentState, **kwargs) -> AgentState:
        ctx = self._build_context(state.reranked_docs or state.fused_docs)
        q = state.normalized_query
        year = state.query_hints.get('_year')  # e.g. 2003; None if no year in query

        if not ctx:
            state.final_answer = 'No supporting context was retrieved.'
            state.evidence_ids = []
            return state

        sents = re.split(r'(?<=[.!?])\s+', ctx)
        q_terms = set(_token_set(q)) - STOP_EN - STOP_DE
        scored = []
        for s in sents:
            s = s.strip()
            if len(s) < 20:
                continue
            st = set(_token_set(s))
            # Base score: normalised query-term overlap * raw match count
            base = len(q_terms & st) / max(len(st), 1) * len(q_terms & st)
            # Temporal bonus: sentence contains the query year or adjacent year
            if year and re.search(r'\b' + str(year) + r'\b', s):
                base *= 2.5
            elif year and any(
                re.search(r'\b' + str(y) + r'\b', s)
                for y in [year - 1, year + 1, year - 2, year + 2]
            ):
                base *= 1.4
            scored.append((base, s))

        scored.sort(key=lambda x: x[0], reverse=True)

        best = []
        seen_terms: set = set()
        for score, s in scored:
            s_terms = set(_token_set(s))
            if seen_terms and len(s_terms & seen_terms) / max(len(s_terms), 1) > 0.6:
                continue
            best.append(s)
            seen_terms |= s_terms
            if len(best) >= 3:
                break

        # If the top-scoring sentence has a zero score, no query terms matched at all —
        # report honestly rather than returning misleading extractive content.
        if not best or scored[0][0] == 0:
            state.final_answer = (
                f'The retrieved context does not contain information about the query'
                + (f' from {year}' if year else '') + '.'
            )
        else:
            state.final_answer = ' '.join(best)

        state.evidence_ids = [_uid(d) for d in (state.reranked_docs or state.fused_docs)[:5] if _uid(d) is not None]
        return state

print('Answer Synthesizer Agent ready.')


Answer Synthesizer Agent ready.


### Explanation: `AnswerSynthesizerAgent`

Purpose: produce an answer from retrieved evidence (mainly extractive, not free-form generation).

Step-by-step:
1. Build context from best docs (`reranked_docs` first, else `fused_docs`).
2. Split context into candidate sentences.
3. Score each sentence by query-term overlap.
4. Add temporal bonus if sentence matches query year.
5. Keep top diverse sentences.
6. Join selected sentences into `state.final_answer`.
7. Save supporting IDs in `state.evidence_ids`.

This keeps answers grounded in retrieved evidence.

In [20]:
# 6) Critic Agent — temporal coherence check for entity_temporal queries
import re as _re2

class CriticAgent(BaseAgent):
    name = 'critic'
    _YEAR_RE = _re2.compile(r'\b(1\d{3}|20\d{2})\b')

    def _temporal_coherent(self, answer: str, query_year: int, window: int = 10) -> bool:
        """Answer must mention at least one year within ±window of the query year.
        Catches answers that are grounded but reference the wrong decade entirely."""
        found = [int(m) for m in self._YEAR_RE.findall(answer)]
        return any(abs(y - query_year) <= window for y in found)

    def run(self, state: AgentState, min_support_overlap: float = 0.45, **kwargs) -> AgentState:
        answer_terms = _token_set(state.final_answer) - STOP_EN - STOP_DE
        if not answer_terms:
            state.critic_ok = False
            state.needs_reretrieval = True
            state.critic_feedback = 'Answer is empty.'
            return state

        support_docs = state.reranked_docs or state.fused_docs
        if not support_docs:
            state.critic_ok = False
            state.needs_reretrieval = True
            state.critic_feedback = 'No support documents available.'
            return state

        support_text = ' '.join(
            (d.metadata.get('original_text') or d.page_content or '') for d in support_docs[:5]
        )
        support_terms = _token_set(support_text) - STOP_EN - STOP_DE
        overlap = len(answer_terms & support_terms) / max(len(answer_terms), 1)

        per_doc_max = 0.0
        for d in support_docs[:5]:
            dt = _token_set(d.metadata.get('original_text') or d.page_content or '') - STOP_EN - STOP_DE
            per_doc_max = max(per_doc_max, len(answer_terms & dt) / max(len(answer_terms), 1))

        grounded = overlap >= min_support_overlap and per_doc_max >= 0.25

        # Temporal coherence: for entity_temporal queries the answer must contain
        # a year within ±10 of the query year.  This catches answers that are
        # lexically grounded (contain 'president', 'ETH') but reference the wrong
        # decade — e.g. returning info about the 2019 president for a 2003 query.
        query_year = state.query_hints.get('_year')
        temporal_ok = True
        temporal_msg = ''
        if state.query_type == 'entity_temporal' and query_year:
            temporal_ok = self._temporal_coherent(state.final_answer, query_year, window=10)
            temporal_msg = (
                f' Temporal check (target={query_year}±10): '
                + ('PASS' if temporal_ok else 'FAIL — answer references wrong time period') + '.'
            )

        state.critic_ok = grounded and temporal_ok
        state.needs_reretrieval = not state.critic_ok
        state.critic_feedback = (
            f'Global overlap={overlap:.3f}, best-doc overlap={per_doc_max:.3f}; '
            f'threshold={min_support_overlap:.2f}/0.25.'
            + temporal_msg
            + (' Grounded.' if state.critic_ok else ' Potentially ungrounded: trigger re-retrieval.')
        )
        return state

print('Critic Agent ready.')


Critic Agent ready.


### Explanation: `CriticAgent`

Purpose: verify whether the generated answer is supported by retrieved evidence.

How it critiques:
1. Build `answer_terms` from answer text (normalized tokens, stopwords removed).
2. Build `support_terms` from top support docs.
3. Compute overlap ratio between answer and support terms.
4. Check thresholds (`min_support_overlap`, `per_doc_max >= 0.25`).
5. For temporal queries, check year consistency (`temporal_ok`).
6. Set `state.critic_ok` and `state.needs_reretrieval`.

Important: `min_support_overlap` is a code heuristic (default `0.45`), not a benchmark-defined value.

In [21]:
# 7) Orchestrator-ready linear execution (can be replaced by external orchestrator)
class MultiAgentPipeline:
    def __init__(self, use_graph: bool = True):
        self.use_graph = use_graph
        self.query_agent = QueryUnderstandingAgent()
        self.bm25_agent = BM25RetrieverAgent(bm25_retriever)
        self.dense_agent = DenseRetrieverAgent(dense_retriever)
        self.graph_agent = GraphRetrieverAgent(graph_retriever)
        self.fusion_agent = FusionAgent()
        self.reranker_agent = ReRankerAgent()
        self.answer_agent = AnswerSynthesizerAgent()
        self.critic_agent = CriticAgent()

    def _safe_retrieval(self, agent, state: AgentState, key: str, top_k: int):
        try:
            return agent.run(state, top_k=top_k)
        except Exception as e:
            state.retrieval_by_agent[key] = []
            state.critic_feedback = (
                (state.critic_feedback + ' ') if state.critic_feedback else ''
            ) + f'{key} retrieval failed: {e}'
            return state

    def _retrieve_all(self, state: AgentState, retrieve_k: int):
        state = self._safe_retrieval(self.bm25_agent, state, 'bm25', retrieve_k)
        state = self._safe_retrieval(self.dense_agent, state, 'dense', retrieve_k)
        if self.use_graph:
            state = self._safe_retrieval(self.graph_agent, state, 'graph', retrieve_k)
        else:
            state.retrieval_by_agent['graph'] = []
        return state

    def run(self, query: str, retrieve_k: int = 50, top_k: int = 5, retry_once: bool = True):
        state = AgentState(query=query)

        # Step A: understanding
        state = self.query_agent.run(state)

        # Step B: retrieval (all three retrievers, isolated failures)
        state = self._retrieve_all(state, retrieve_k)

        # Step C: fusion -> rerank -> synthesis -> critique
        state = self.fusion_agent.run(state, top_k=retrieve_k)
        state = self.reranker_agent.run(state, top_k=top_k)
        state = self.answer_agent.run(state)
        state = self.critic_agent.run(state)

        # Retry with boosted weights if critic flags the answer as ungrounded
        if retry_once and state.needs_reretrieval:
            print('[Pipeline] Critic triggered re-retrieval...')
            boosted = dict(state.query_hints)
            boosted['bm25'] = boosted.get('bm25', 1.0) + 0.3
            boosted['dense'] = boosted.get('dense', 1.0) + 0.3
            state.query_hints = boosted
            state = self._retrieve_all(state, retrieve_k)
            state = self.fusion_agent.run(state, top_k=retrieve_k)
            state = self.reranker_agent.run(state, top_k=top_k)
            state = self.answer_agent.run(state)
            state = self.critic_agent.run(state)

        # If the critic still disagrees after all attempts, be honest rather
        # than returning a plausible-sounding but wrong extractive answer.
        if not state.critic_ok:
            year_hint = state.query_hints.get('_year')
            year_str  = f' from {year_hint}' if year_hint else ''
            state.final_answer = (
                f'The available corpus does not contain sufficient evidence to '
                f'reliably answer this query{year_str}. Retrieved context covers '
                f'related topics but does not directly address the question.'
            )
        return state

pipeline = MultiAgentPipeline(use_graph=True)
print('Multi-agent pipeline ready. Graph enabled:', pipeline.use_graph)


Multi-agent pipeline ready. Graph enabled: True


### Explanation: `MultiAgentPipeline` (sequential orchestration)

This is the core execution flow.

Flow:
1. `QueryUnderstandingAgent` classifies query + sets weights.
2. Retrieve from BM25/Dense/Graph agents.
3. `FusionAgent` merges rankings (RRF + dedupe).
4. `ReRankerAgent` improves top ordering.
5. `AnswerSynthesizerAgent` composes answer from evidence.
6. `CriticAgent` checks grounding/temporal consistency.
7. If critic fails, run one retry with boosted retrieval weights.
8. If still unsupported, return an honest fallback answer.

This is a sequential pipeline strategy and can be compared to other strategies in Step 3.

In [22]:
# Example usage
sample_query = 'Who were the rectors of ETH between 2017 and 2022?'
out = pipeline.run(sample_query, retrieve_k=20, top_k=5, retry_once=True)

print('Query type:', out.query_type)
print('Query hints:', {k: v for k, v in out.query_hints.items() if not str(k).startswith('_')})
print('Critic OK:', out.critic_ok)
print('Critic feedback:', out.critic_feedback)
print('Evidence IDs:', out.evidence_ids[:5])
print('\nFinal answer:\n', out.final_answer)


[ReRanker] Loading CrossEncoder (ms-marco-MiniLM-L-6-v2)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[ReRanker] CrossEncoder ready.
Query type: entity_temporal
Query hints: {'bm25': 0.7, 'dense': 1.1, 'graph': 1.4}
Critic OK: True
Critic feedback: Global overlap=1.000, best-doc overlap=0.561; threshold=0.45/0.25. Temporal check (target=2017±10): PASS. Grounded.
Evidence IDs: ['73afc89edd471ff98176f33babf28b62dccf4ac6_fixed_2', '00859327fafc62821b53b9ad083e2a244c1b4470_fixed_3', '20aed32900c864b6d99d046fa3706c8dec5194d6_fixed_0', 'b7471c1deec222c6c0529cfaf9285b9c075c5096_fixed_1', 'ce4529dd92b5b28f5b642e7a04f73206e38024f0_fixed_1']

Final answer:
 the [vice rector for continuing education](https://ethz.ch/en/the-eth-zurich/organisation/vice-rectors.html) assists the rector in the area of continuing education for those with an academic background at eth zurich. reaching critical mass of women at eth: /equal-opportunities/strategie-und-zahlen/frauen-an-der-eth/frauenstreik-eth/promoting-women.html) **sarah springman** professor of geotechnical engineering at eth zurich and rector of eth 

In [23]:
# Retriever debug view: inspect top results before and after fusion

def _preview_docs(name, docs, n=3, max_chars=220):
    print(f"\n[{name}] top {min(n, len(docs))} / {len(docs)}")
    for i, d in enumerate(docs[:n], start=1):
        uid = _uid(d)
        text = (d.metadata.get('original_text') or d.page_content or '').replace('\n', ' ').strip()
        text = text[:max_chars] + ('...' if len(text) > max_chars else '')
        score_keys = [k for k in ('bm25_score', 'dense_score', 'grag_score', 'fused_score', 'rerank_score') if k in d.metadata]
        score_str = ', '.join(f"{k}={d.metadata.get(k):.4f}" for k in score_keys)
        print(f"{i}. id={uid} | {score_str}")
        print(f"   {text}")


def debug_query(query, retrieve_k=50, top_k=5, include_graph=True):
    print('Query:', query)
    print('include_graph:', include_graph)

    # Individual retrievers
    bm = _safe_unique(bm25_retriever.search(query, top_k=retrieve_k))
    de = _safe_unique(dense_retriever.search(query, top_k=retrieve_k))
    gr = _safe_unique(graph_retriever.search(query, top_k=retrieve_k)) if include_graph else []

    _preview_docs('BM25', bm)
    _preview_docs('Dense', de)
    if include_graph:
        _preview_docs('GraphRAG', gr)

    # Full pipeline outputs
    dbg_pipeline = MultiAgentPipeline(use_graph=include_graph)
    out = dbg_pipeline.run(query, retrieve_k=retrieve_k, top_k=top_k, retry_once=True)
    _preview_docs('Fused', out.fused_docs)
    _preview_docs('Re-ranked', out.reranked_docs)

    print('\nQuery type:', out.query_type)
    print('Query hints:', out.query_hints)
    print('Critic OK:', out.critic_ok)
    print('Critic feedback:', out.critic_feedback)
    print('Evidence IDs:', out.evidence_ids)
    print('\nFinal answer:\n', out.final_answer)

# Example debug call:
debug_query('Who were the rectors of ETH between 2017 and 2022?', retrieve_k=20, top_k=5, include_graph=True)


Query: Who were the rectors of ETH between 2017 and 2022?
include_graph: True

[BM25] top 3 / 20
1. id=16f49b83f2f6f702c29efa8987cb10c0ce68b7c7_fixed_0 | bm25_score=21.0650, fused_score=0.0115
   eth zurich at wef 2017: international exchange the eth delegation is also using the world economic forum 2017 as an opportunity for exchange with the huge range of wef participants from around the world. there are numero...
2. id=8adacd9ad5e2ee98d5e97dfe98c7b2587a0694ba_fixed_0 | bm25_score=20.9375, fused_score=0.0113
   challenge the best in data science: eth zurich's continuing education offensive is gathering pace. after launching the [school for continuing education](/en/news-and-events/eth-news/news/2018/04/continuing-education.html...
3. id=1afe6e1de8c417b8ea5406f8d9b529ebf22966ef_fixed_0 | bm25_score=20.9113, fused_score=0.0111
   what previous bird flu outbreaks teach us: - the bird flu epidemic in china from 2013 to 2017 showed that pathogens can circulate in poultry farms for several

### Explanation: Debug helper cell

This cell is for diagnosis, not main pipeline logic.

Step-by-step:
1. Print top docs from each retriever (`BM25`, `Dense`, `GraphRAG`).
2. Run full pipeline on the same query.
3. Print fused and reranked docs with scores.
4. Print trace info (`query_type`, `query_hints`, `critic_feedback`, `evidence_ids`).

Use this when answers are wrong to find exactly which stage fails.

## Step 3 Implementation Plan (Strategy A: Confidence-Based Routing)

This section is a guided build plan after the Step 2 base pipeline is ready.

**Why Confidence-Based Routing for Strategy A:**
The base pipeline (cell `MultiAgentPipeline`) already does *Parallel + Fusion* with one critic retry — this is essentially the Voting baseline. To deliver a **distinct, swappable** orchestration strategy, we build a Confidence-Based router that:
- uses `QueryUnderstandingAgent` for **classification only** (`query_type`),
- lets the **orchestrator own the weight presets** (configurable, not buried inside the agent),
- can optionally **gate out** retrievers entirely (set weight to 0 / skip the call) when confidence is low,
- emits a **trace dict** for explainability (which preset, which retrievers ran, retry count).

**Goal:**
- Implement Confidence-Based Routing cleanly as a swappable orchestrator class.
- Evaluate with required metrics (quantitative, qualitative, efficiency) on the same QA set used in Step 1 / baseline.
- Keep structure modular so we can swap in Person A's refined agents at the Week 2 integration checkpoint.

**How to use this plan:**
1. Read one markdown cell.
2. Implement only that part in the next code cell.
3. Run and verify before moving on.
4. Keep changes small and traceable — one concept per cell.

---

### Overall implementation structure (Mermaid)

```mermaid
flowchart TD
    A[Step 2 Base Ready] --> B[Step 1 Config Cell]
    B --> C[Step 2 Optional Bilingual Helper]
    C --> D[Step 3 Confidence Orchestrator]
    D --> E[Step 4 Single-Query Smoke Test]
    E --> F[Step 5 Batch Quantitative Eval<br/>P@k / R@k / MRR / nDCG]
    F --> G[Step 6 Efficiency Eval<br/>Latency + Retry Count + Gated Calls]
    G --> H[Step 7 Qualitative Eval<br/>Explainability + Complementarity + Failure]
    H --> I[Step 8 Comparative Analysis vs Voting baseline]
    I --> J[Step 9 Charts + Tables + Export]
    J --> K[Checklist + Team Handoff]

    subgraph Confidence Core Flow
        D1[QueryUnderstanding<br/>classify only] --> D2[Pick Weight Preset<br/>by query_type]
        D2 --> D3[Optional Gating<br/>skip low-weight retrievers]
        D3 --> D4[Run Selected Retrievers]
        D4 --> D5[Weighted RRF Fusion]
        D5 --> D6[Re-Rank]
        D6 --> D7[Synthesize Answer]
        D7 --> D8[Critic Check]
        D8 --> D9{critic_ok?}
        D9 -- No --> D10[Retry once<br/>broaden weights]
        D10 --> D4
        D9 -- Yes --> D11[Return answer + trace]
    end
```

> **This-week scope (Dongyuan, Week 1–2):** Steps 1 → 4 (config, optional bilingual helper, orchestrator, smoke test). Steps 5–9 are Week 3+ when Julia leads evaluation.

In [24]:
# === Step 1 — Strategy A configuration ===
# All tunable knobs for Confidence-Based Routing live here.
# The orchestrator reads ONLY from this config, so swapping presets,
# thresholds, or feature flags never requires touching the pipeline code.
# This is what makes the strategy "swappable" and easy to ablate later.

# (1) Weight presets per query_type.
# These are the values that QueryUnderstandingAgent ALSO computes internally,
# but for Confidence routing we want the orchestrator (not the agent) to own
# them — that way Julia's Waterfall / Voting can reuse the same agents with
# completely different routing logic.
WEIGHT_PRESETS = {
    'entity_temporal': {'bm25': 0.7, 'dense': 1.1, 'graph': 1.4},
    'entity':          {'bm25': 0.9, 'dense': 1.1, 'graph': 1.3},
    'keyword':         {'bm25': 1.3, 'dense': 1.1, 'graph': 0.7},
    'semantic':        {'bm25': 0.8, 'dense': 1.4, 'graph': 0.9},
    'graph':           {'bm25': 0.8, 'dense': 1.0, 'graph': 1.4},
    'mixed':           {'bm25': 1.0, 'dense': 1.2, 'graph': 0.8},
}

# (2) Gating threshold — when a retriever's preset weight is BELOW this
# value, we skip the retrieval call entirely (saves real latency).
# Set to 0.0 to disable gating and get plain weighted Voting behaviour.
GATE_THRESHOLD = 0.75

# (3) Feature flags
USE_BILINGUAL_QUERY = False   # toggle the Step 2 EN<->DE helper
USE_GATING          = True    # apply GATE_THRESHOLD
USE_RETRY           = True    # one critic-driven retry with broadened weights

# (4) Evaluation knobs — must match the baseline report so tables are comparable.
K_VALUES   = (1, 3, 5, 10)    # same k as baseline_repro_report.md
RETRIEVE_K = 50               # how many docs each retriever returns before fusion
TOP_K      = 10               # final ranked list size

print('[Config] Confidence routing config loaded.')
print('  WEIGHT_PRESETS keys :', list(WEIGHT_PRESETS.keys()))
print('  GATE_THRESHOLD      :', GATE_THRESHOLD)
print('  USE_GATING          :', USE_GATING)
print('  USE_RETRY           :', USE_RETRY)
print('  USE_BILINGUAL_QUERY :', USE_BILINGUAL_QUERY)
print('  K_VALUES            :', K_VALUES)

[Config] Confidence routing config loaded.
  WEIGHT_PRESETS keys : ['entity_temporal', 'entity', 'keyword', 'semantic', 'graph', 'mixed']
  GATE_THRESHOLD      : 0.75
  USE_GATING          : True
  USE_RETRY           : True
  USE_BILINGUAL_QUERY : False
  K_VALUES            : (1, 3, 5, 10)


### Step 1 — Strategy config cell

**What to implement in the next code cell:**
A small config block holding all tunable values for Strategy A. Centralizing these makes evaluation fair and tuning transparent.

**Essential config items:**
- `RETRIEVE_K` — how many docs each retriever returns (e.g. 30).
- `TOP_K` — final number returned after rerank (e.g. 10).
- `K_RRF` — RRF smoothing constant (e.g. 60).
- `WEIGHT_PRESETS` — dict mapping `query_type` → `{bm25, dense, graph}` weights. This is the **core of Confidence routing** — the orchestrator picks one preset per query.
- `GATE_THRESHOLD` — minimum weight below which a retriever is **skipped entirely** (saves latency). Set to `0.0` to disable gating.
- `K_VALUES` — evaluation cutoffs (e.g. `[1, 3, 5, 10]`).
- Flags: `USE_BILINGUAL_QUERY`, `USE_RETRY`, `USE_GATING`.
- Benchmark paths: `PATH_QRELS`, `PATH_QUERIES` (needed by Step 5+).

**Suggested default `WEIGHT_PRESETS` (matches `QueryUnderstandingAgent` types):**
```
entity_temporal : {bm25: 0.7, dense: 1.1, graph: 1.4}   # graph community search wins on person-role-year
entity          : {bm25: 0.9, dense: 1.1, graph: 1.3}
keyword         : {bm25: 1.3, dense: 1.1, graph: 0.7}   # exact-match queries lean BM25
semantic        : {bm25: 0.8, dense: 1.4, graph: 0.9}   # paraphrase queries lean Dense
graph           : {bm25: 0.8, dense: 1.0, graph: 1.4}
mixed           : {bm25: 1.0, dense: 1.2, graph: 0.8}   # safe default
```

**Reason:**
- A config cell prevents hidden magic numbers and keeps comparisons against Voting/Waterfall fair.
- `WEIGHT_PRESETS` lives in config (not inside the agent) so we can ablate easily in Step 8.
- Including benchmark paths here means evaluation cells later don't have to re-discover them.

**Learning point:** This is the *only* place where weights live. If Julia ablates them later, she changes one cell — not five.

In [25]:
# === Step 2 — Static bilingual keyword expander (NOT a translator) ===
#
# Purpose: a zero-dependency ablation lever. When enabled via
# USE_BILINGUAL_QUERY, each query is paired with a second "variant" where
# a handful of ETH-domain terms are swapped between DE and EN. Retrievers
# are then run on BOTH variants and the results are merged (dedup by doc id).
#
# IMPORTANT honest labelling:
#   * This is NOT an NLP translation model.
#   * It is a regex-based substitution over ~20 hand-picked term pairs.
#   * Grammar, compounds, word sense, and polysemy are NOT handled.
#   * The dense retriever (multilingual-e5-large-instruct) already does
#     cross-language matching at the embedding level, so real MT would
#     mostly be redundant here. See the markdown cell above for the full
#     design-trade-off note.
#
# When to enable:
#   * Only when you want a small "with/without helper" data point in the
#     Step 8 ablation table. Keep OFF by default.

DE_EN_TERM_MAP = {
    'rektor': 'rector', 'rektorin': 'rector',
    'praesident': 'president', 'präsident': 'president',
    'forschung': 'research', 'studierende': 'students',
    'professor': 'professor', 'institut': 'institute',
    'jahr': 'year', 'zwischen': 'between',
    'wer': 'who', 'wann': 'when',
}
EN_DE_TERM_MAP = {
    'rector': 'rektor', 'president': 'präsident',
    'research': 'forschung', 'students': 'studierende',
    'between': 'zwischen', 'year': 'jahr',
    'who': 'wer', 'when': 'wann',
}


def make_query_variants(query: str, enable: bool = USE_BILINGUAL_QUERY):
    """Return [original] or [original, expanded_variant].

    The ORIGINAL query is always first so we never lose retrieval quality —
    the expanded variant is purely additive (any extra hits it finds are merged
    via deduplicated union in ConfidenceOrchestrator._retrieve).
    """
    if not enable or not query:
        return [query]
    q_low = query.lower()

    has_de = any(re.search(r'\b' + de + r'\b', q_low) for de in DE_EN_TERM_MAP)
    has_en = any(re.search(r'\b' + en + r'\b', q_low) for en in EN_DE_TERM_MAP)

    variants = [query]
    if has_de and not has_en:
        expanded = q_low
        for de, en in DE_EN_TERM_MAP.items():
            expanded = re.sub(r'\b' + de + r'\b', en, expanded)
        if expanded != q_low:
            variants.append(expanded)
    elif has_en and not has_de:
        expanded = q_low
        for en, de in EN_DE_TERM_MAP.items():
            expanded = re.sub(r'\b' + en + r'\b', de, expanded)
        if expanded != q_low:
            variants.append(expanded)
    return variants


# Quick sanity check — should emit 2 variants for both directions when enabled.
print(make_query_variants('Who was the rector in 2003?', enable=True))
print(make_query_variants('Wer war der Rektor 2003?',     enable=True))
print(make_query_variants('Who was the rector in 2003?', enable=False))

['Who was the rector in 2003?', 'wer was the rektor in 2003?']
['Wer war der Rektor 2003?', 'who war der rector 2003?']
['Who was the rector in 2003?']


### Step 2 — Optional bilingual keyword expander (not a translator)

**What this is — honest framing:**
A tiny hand-written EN↔DE keyword map (~20 pairs) that regex-substitutes common ETH-domain terms (Rektor ↔ rector, Forschung ↔ research, …). It is a **zero-dependency lexical helper**, *not* a machine translation model.

**Why not a real MT model (e.g. Helsinki-NLP `opus-mt-de-en`)?**
- The dense retriever in this notebook already uses **`multilingual-e5-large-instruct`**, a multilingual sentence-embedding model. That model is specifically designed to match a DE query against an EN document (and vice versa) in vector space *without* any translation step.
- So real MT would mostly duplicate work the dense retriever already does, while adding ~300 MB model download, extra latency, and risk of mistranslations that confuse BM25.
- The one place MT *could* help is BM25 (which is purely lexical and language-blind). For the ~2 German queries in the 24-query benchmark, the expected MRR gain is small (0.00–0.02) and not worth the engineering cost for this project.

**Why keep the dictionary helper at all?**
- It is an honest, cheap **ablation lever**: flip `USE_BILINGUAL_QUERY` on/off in Step 1 and measure the delta in Step 8. This becomes a small but defensible table row in the report.
- It costs nothing when disabled (which is the default).

**Design choice:**
- Default is `USE_BILINGUAL_QUERY = False`. Confidence routing's main MRR lever is **weight-preset + gating tuning**, not translation.
- We document this decision in the code cell below so the trade-off is reviewable.

In [26]:
# === Step 3 — Confidence-Based Orchestrator ===
# A swappable orchestrator that:
#   1. Classifies the query (QueryUnderstandingAgent)
#   2. Picks a weight preset from WEIGHT_PRESETS (orchestrator owns weights)
#   3. Optionally GATES retrievers below GATE_THRESHOLD (skips the call entirely)
#   4. Runs the surviving retrievers on the query (+ optional bilingual variants)
#   5. Weighted RRF fusion with the chosen weights
#   6. Cross-encoder re-rank
#   7. Extractive answer synthesis
#   8. Critic check; on failure, ONE retry with broadened weights + no gating
#   9. Returns (answer, top_docs, trace) — trace dict drives explainability.

import time

class ConfidenceOrchestrator:
    def __init__(self,
                 weight_presets=None,
                 gate_threshold: float = GATE_THRESHOLD,
                 use_gating: bool = USE_GATING,
                 use_retry: bool = USE_RETRY,
                 use_bilingual: bool = USE_BILINGUAL_QUERY):
        self.weight_presets = weight_presets or WEIGHT_PRESETS
        self.gate_threshold = gate_threshold
        self.use_gating     = use_gating
        self.use_retry      = use_retry
        self.use_bilingual  = use_bilingual

        # Reuse the agents already defined earlier in this notebook.
        self.query_agent    = QueryUnderstandingAgent()
        self.retrievers = {
            'bm25':  BM25RetrieverAgent(bm25_retriever),
            'dense': DenseRetrieverAgent(dense_retriever),
            'graph': GraphRetrieverAgent(graph_retriever),
        }
        self.fusion_agent   = FusionAgent()
        self.reranker_agent = ReRankerAgent()
        self.answer_agent   = AnswerSynthesizerAgent()
        self.critic_agent   = CriticAgent()

    # ---------- helpers ----------
    def _select_preset(self, query_type: str):
        # Fall back to 'mixed' if QueryUnderstandingAgent emits an unseen type.
        return dict(self.weight_presets.get(query_type, self.weight_presets['mixed']))

    def _apply_gating(self, weights: dict):
        """Drop retrievers with weight < threshold. Never gate everything out."""
        if not self.use_gating:
            return dict(weights), []
        gated = [n for n, w in weights.items() if w < self.gate_threshold]
        kept  = {n: w for n, w in weights.items() if w >= self.gate_threshold}
        if not kept:                       # safety: keep at least one retriever
            return dict(weights), []
        return kept, gated

    def _retrieve(self, state: AgentState, active_names, retrieve_k: int,
                  query_variants):
        """Run each active retriever once per query variant, dedupe, and store
        the merged hits back into state.retrieval_by_agent."""
        if not hasattr(state, 'retrieval_errors'):
            state.retrieval_errors = {}
        if not hasattr(state, 'zero_result_retrievers'):
            state.zero_result_retrievers = []

        for name in active_names:
            agent = self.retrievers[name]
            merged, seen = [], set()
            had_error = None
            for variant in query_variants:
                state.normalized_query = variant
                try:
                    agent.run(state, top_k=retrieve_k)
                except Exception as e:
                    had_error = repr(e)
                    state.retrieval_by_agent[name] = []
                    continue
                # agent.run() just OVERWROTE state.retrieval_by_agent[name]
                # with THIS variant's results. We read it immediately to
                # extract any new unique docs before the next variant runs
                # (otherwise the next variant would clobber them).
                for d in state.retrieval_by_agent.get(name, []):
                    u = _uid(d)
                    if u and u not in seen:
                        seen.add(u); merged.append(d)
            state.retrieval_by_agent[name] = merged
            if merged:
                state.retrieval_errors.pop(name, None)
                if name in state.zero_result_retrievers:
                    state.zero_result_retrievers.remove(name)
            else:
                if name not in state.zero_result_retrievers:
                    state.zero_result_retrievers.append(name)
                state.retrieval_errors[name] = had_error or 'retriever returned 0 unique docs'
        # Ensure inactive retrievers exist as empty lists so fusion math is clean.
        for name in self.retrievers:
            state.retrieval_by_agent.setdefault(name, [])
        return state

    # ---------- main entry ----------
    def run(self, query: str,
            retrieve_k: int = RETRIEVE_K,
            top_k: int = TOP_K):
        t0 = time.time()
        state = AgentState(query=query)
        state.retrieval_errors = {}
        state.zero_result_retrievers = []

        # 1) classify
        state = self.query_agent.run(state)
        original_norm = state.normalized_query

        # 2) pick preset (overrides whatever the agent put in query_hints)
        preset_weights = self._select_preset(state.query_type)

        # 3) gating
        active_weights, gated_out = self._apply_gating(preset_weights)
        active_names = list(active_weights.keys())

        # 4) retrieval (optionally bilingual)
        variants = make_query_variants(original_norm, enable=self.use_bilingual)
        state = self._retrieve(state, active_names, retrieve_k, variants)
        state.normalized_query = original_norm        # restore for fusion / answer

        # 5) fusion uses ONLY the active weights (gated retrievers are absent)
        state.query_hints = {**state.query_hints, **active_weights}
        state = self.fusion_agent.run(state, top_k=retrieve_k)

        # 6-8) rerank, synthesize, critic
        state = self.reranker_agent.run(state, top_k=top_k)
        state = self.answer_agent.run(state)
        state = self.critic_agent.run(state)

        retry_triggered = False
        retry_weights = None
        if self.use_retry and state.needs_reretrieval:
            retry_triggered = True
            # Broaden: include all retrievers, lift every weight to >= 1.0,
            # and disable gating for this retry pass.
            broadened = {n: max(preset_weights.get(n, 1.0), 1.0)
                         for n in self.retrievers}
            retry_weights = dict(broadened)
            state = self._retrieve(state, list(broadened.keys()),
                                   retrieve_k, variants)
            state.query_hints = {**state.query_hints, **broadened}
            state = self.fusion_agent.run(state, top_k=retrieve_k)
            state = self.reranker_agent.run(state, top_k=top_k)
            state = self.answer_agent.run(state)
            state = self.critic_agent.run(state)

        latency = time.time() - t0

        # Trace dict — this is the explainability deliverable.
        trace = {
            'query':            query,
            'query_type':       state.query_type,
            'weights':          active_weights,
            'retry_weights':    retry_weights,
            'gated_out':        gated_out,
            'retriever_counts': {n: len(state.retrieval_by_agent.get(n, []))
                                 for n in self.retrievers},
            'retrieval_errors': dict(getattr(state, 'retrieval_errors', {})),
            'zero_result_retrievers': list(getattr(state, 'zero_result_retrievers', [])),
            'retry_triggered':  retry_triggered,
            'critic_ok':        state.critic_ok,
            'critic_feedback':  state.critic_feedback,
            'evidence_ids':     state.evidence_ids,
            'latency_s':        round(latency, 4),
            'bilingual_variants': len(variants),
        }
        top_docs = (state.reranked_docs or state.fused_docs)[:top_k]
        return state.final_answer, top_docs, trace


orchestrator = ConfidenceOrchestrator()
print('Confidence orchestrator ready.')

Confidence orchestrator ready.


### Step 3 — Build the Confidence-Based orchestrator

**What to implement in the next code cell:**
A `ConfidenceOrchestrator` class (or `confidence_orchestrate(...)` function) that exposes a single entry point and returns `(answer, docs, trace)`.

**Responsibilities (in order):**
1. **Classify query** — call `QueryUnderstandingAgent` to get `query_type` (we ignore its weight hints; the orchestrator owns weights).
2. **Pick weight preset** — look up `WEIGHT_PRESETS[query_type]` from config; fall back to `mixed`.
3. **Apply gating** (if `USE_GATING`) — drop any retriever whose weight is `< GATE_THRESHOLD`. Gated retrievers are **not called at all** (saves latency).
4. **Run selected retrievers** — for each query variant returned by the bilingual helper, call only the surviving retriever agents.
5. **Weighted RRF fusion** — pass the chosen weights into `FusionAgent` via `state.query_hints`.
6. **Re-rank** — `ReRankerAgent.run(state, top_k=TOP_K)`.
7. **Synthesize answer** — `AnswerSynthesizerAgent.run(state)`.
8. **Critic check** — `CriticAgent.run(state)`. If `state.needs_reretrieval` and `USE_RETRY`, do **one retry** with broadened weights (e.g. set all weights to ≥1.0 and disable gating for retry).
9. **Return** `state.final_answer`, `state.reranked_docs`, and a **trace dict**.

**Trace dict (for explainability + debug):**
```
{
  'query_type'     : 'entity_temporal',
  'weights'        : {'bm25': 0.7, 'dense': 1.1, 'graph': 1.4},
  'gated_out'      : [],                    # retrievers skipped this turn
  'retriever_counts': {'bm25': 30, 'dense': 30, 'graph': 30},
  'fused_n'        : 67,
  'reranked_n'     : 10,
  'retry_triggered': False,
  'critic_ok'      : True,
  'evidence_ids'   : [...]
}
```

**Reason:**
- Trace makes the orchestrator **explainable** (Stretch Goal A in plan) almost for free.
- Owning weight presets at the orchestrator level (not inside `QueryUnderstandingAgent`) is what makes this a true *swappable* strategy — Julia's Waterfall / Voting can reuse the same agents with different routing logic.
- The retry policy is intentionally simple (one broadened pass) so we don't conflate Confidence routing with a full Critic Loop.

**Learning point:** Keep the orchestrator a *thin* class. Each step is one method call on an existing agent. If you find yourself reimplementing retrieval logic inside the orchestrator, that logic belongs in the agent.

In [27]:
# === Step 4 — Smoke test: 3 representative queries ===
# We pick one query per major type so the trace clearly shows
#   (a) different presets fire,
#   (b) gating actually drops some retrievers, and
#   (c) the critic pass/retry behaves as expected.
import json as _json

smoke_queries = [
    # entity_temporal -> graph-heavy preset, BM25 likely gated
    'Who were the rectors of ETH between 2017 and 2022?',
    # keyword/factual -> BM25-heavy preset, graph likely gated
    'Define what a Habilitation is at ETH.',
    # semantic -> dense-heavy preset
    'How does sustainability research connect with industry partnerships at ETH?',
]

for q in smoke_queries:
    print('\n' + '=' * 80)
    print('QUERY:', q)
    answer, docs, trace = orchestrator.run(q, retrieve_k=20, top_k=5)
    # Print compact trace (skip long critic feedback so the cell stays readable)
    compact = {k: v for k, v in trace.items() if k != 'critic_feedback'}
    print('TRACE:')
    print(_json.dumps(compact, indent=2, default=str))
    print('CRITIC :', trace['critic_feedback'])
    print('ANSWER :', (answer or '')[:400])


QUERY: Who were the rectors of ETH between 2017 and 2022?
TRACE:
{
  "query": "Who were the rectors of ETH between 2017 and 2022?",
  "query_type": "entity_temporal",
  "weights": {
    "dense": 1.1,
    "graph": 1.4
  },
  "retry_weights": null,
  "gated_out": [
    "bm25"
  ],
  "retriever_counts": {
    "bm25": 0,
    "dense": 20,
    "graph": 20
  },
  "retrieval_errors": {},
  "zero_result_retrievers": [],
  "retry_triggered": false,
  "critic_ok": true,
  "evidence_ids": [
    "73afc89edd471ff98176f33babf28b62dccf4ac6_fixed_2",
    "00859327fafc62821b53b9ad083e2a244c1b4470_fixed_3",
    "20aed32900c864b6d99d046fa3706c8dec5194d6_fixed_0",
    "b7471c1deec222c6c0529cfaf9285b9c075c5096_fixed_1",
    "ce4529dd92b5b28f5b642e7a04f73206e38024f0_fixed_1"
  ],
  "latency_s": 0.1369,
  "bilingual_variants": 1
}
CRITIC : Global overlap=1.000, best-doc overlap=0.561; threshold=0.45/0.25. Temporal check (target=2017±10): PASS. Grounded.
ANSWER : the [vice rector for continuing education](htt

### Step 4 — Quick smoke test (single query)

**What to implement in the next code cell:**
Run **3 representative queries** through the orchestrator — one per major query type — and print the trace + final answer for each. Recommended set:
- An `entity_temporal` query, e.g. `"Who was president of ETH in 2003?"`
- A `keyword` query, e.g. `"When was ETH Zurich founded?"`
- A `semantic` query, e.g. `"How does ETH support sustainable research?"`

**For each query print:**
- `query_type`
- `weights` chosen by the preset
- `gated_out` retrievers (if any)
- `retry_triggered`
- `critic_ok`
- top 3 evidence IDs
- the final answer (truncated to ~300 chars)

**Reason:**
- Catches pipeline issues early before the full benchmark loop in Step 5.
- Confirms the **routing actually changes behavior** (different presets must produce visibly different evidence IDs) — otherwise the strategy is no different from the Voting baseline.

**Learning point:** If the trace output looks identical across query types, the weights aren't reaching the fusion step. Common cause: forgetting to pass weights into `state.query_hints` before calling `FusionAgent.run(state)`.

### Step 5 — Quantitative evaluation loop

**What to implement in the next code cell:**
Loop over the benchmark QA queries (24 questions) and build a `pytrec_eval`-style run dict for Strategy A.

**Compute and report:**
- `Precision@k` for `k ∈ K_VALUES`
- `Recall@k` for `k ∈ K_VALUES`
- `MRR`
- `nDCG@k` (recommended — gives ranking-quality signal that P/R miss)

**Output:**
- A **per-query** table (24 rows × metrics) — useful later for failure analysis.
- A **summary** table (single row of averaged metrics).

**Reason:**
- These metrics are mandatory Step 3 deliverables and let us compare directly against the Step 1 baseline numbers.
- Per-query scores feed directly into Step 7 (failure analysis) and Step 8 (paired t-test vs baseline).

**Learning point:** Use the **same `K_VALUES` and same qrels file** as the Step 1 baseline report. Otherwise the comparison numbers are not meaningful.

In [28]:
# === Step 5 — Quantitative evaluation ===
# Loads the SAME QA file and qrels folder as the baseline report and produces
# a summary table with EXACTLY the same column ordering:
#   queries_evaluated, MRR, Precision@1, Recall@1, P@3, R@3, P@5, R@5, P@10, R@10
# This makes side-by-side comparison with baseline_repro_report.md trivial
# (just append the new row).

import json
import statistics
import pandas as pd
from collections import defaultdict

QA_PATH   = PROJECT_ROOT / 'benchmark' / 'benchmark_qa_bilingual.json'
QRELS_DIR = PROJECT_ROOT / 'benchmark' / 'score' / 'fixed_size'
assert QA_PATH.exists(),   f'QA file not found at {QA_PATH}'
assert QRELS_DIR.exists(), f'Qrels dir not found at {QRELS_DIR}'


def load_qrels(qrels_dir, threshold: float = 0.5):
    """Read per-doc qrel files. File name = doc_id, content = {qid: {relevance_score: x}}.
    A (qid, doc_id) pair counts as relevant when relevance_score >= threshold.
    Same loader pattern as `baseline_repro_runner.py`.
    """
    qrels = defaultdict(set)
    for fp in sorted(qrels_dir.glob('*.json')):
        doc_id = fp.stem
        payload = json.loads(fp.read_text(encoding='utf-8'))
        for qid, info in payload.items():
            if float(info.get('relevance_score', 0.0)) >= threshold:
                qrels[str(qid)].add(doc_id)
    return qrels


def precision_recall_at_k(ranked, relevant, k):
    top_k = ranked[:k]
    if not top_k:
        return 0.0, 0.0
    hit = sum(1 for d in top_k if d in relevant)
    return hit / k, hit / max(len(relevant), 1)


def reciprocal_rank(ranked, relevant):
    for i, d in enumerate(ranked, start=1):
        if d in relevant:
            return 1.0 / i
    return 0.0


def evaluate_run(method_name: str, run_dict: dict, qrels_dict: dict,
                 k_values=K_VALUES):
    """Compute the standard IR summary + per-query rows for one run.
    Output column ordering is intentionally identical to baseline_repro_report.md."""
    qids = sorted(set(qrels_dict.keys()) & set(run_dict.keys()))
    rr_vals = []
    p_vals  = {k: [] for k in k_values}
    r_vals  = {k: [] for k in k_values}
    per_q   = []
    for qid in qids:
        ranked = run_dict[qid]
        rel    = qrels_dict[qid]
        rr     = reciprocal_rank(ranked, rel)
        rr_vals.append(rr)
        row = {'method': method_name, 'qid': qid, 'MRR': rr}
        for k in k_values:
            p, r = precision_recall_at_k(ranked, rel, k)
            p_vals[k].append(p); r_vals[k].append(r)
            row[f'Precision@{k}'] = p
            row[f'Recall@{k}']    = r
        per_q.append(row)
    summary = {'method': method_name,
               'queries_evaluated': len(qids),
               'MRR': statistics.fmean(rr_vals) if rr_vals else 0.0}
    for k in k_values:
        summary[f'Precision@{k}'] = statistics.fmean(p_vals[k]) if p_vals[k] else 0.0
        summary[f'Recall@{k}']    = statistics.fmean(r_vals[k]) if r_vals[k] else 0.0
    return summary, pd.DataFrame(per_q)


# ---- Build the run for Strategy A ----
qa_data = json.loads(QA_PATH.read_text(encoding='utf-8'))
qrels   = load_qrels(QRELS_DIR)
eval_qa_data = [item for item in qa_data if str(item['id']) in qrels]
print(f'Loaded {len(qa_data)} QA items and qrels for {len(qrels)} qids.')
print(f'Evaluating {len(eval_qa_data)} QA items that have qrels.')

run_confidence       = {}      # qid -> [doc_id, ...]
traces_confidence    = {}      # qid -> trace dict (for Steps 6/7)
latencies_confidence = {}      # qid -> seconds (for Steps 6/8)

for item in eval_qa_data:
    qid   = str(item['id'])
    query = item.get('question') or item.get('query')
    answer, docs, trace = orchestrator.run(query,
                                           retrieve_k=RETRIEVE_K,
                                           top_k=max(K_VALUES))
    run_confidence[qid]       = [_uid(d) for d in docs if _uid(d) is not None]
    traces_confidence[qid]    = trace
    latencies_confidence[qid] = trace['latency_s']

summary_conf, perq_conf = evaluate_run('Confidence (A)', run_confidence, qrels)

# Format the row in the SAME column order as the baseline report
ordered_cols = ['method', 'queries_evaluated', 'MRR']
for k in K_VALUES:
    ordered_cols += [f'Precision@{k}', f'Recall@{k}']
summary_df = pd.DataFrame([summary_conf])[ordered_cols]
print('\n=== Strategy A — Confidence-Based Routing — quantitative summary ===')
print(summary_df.to_string(index=False))

Loaded 25 QA items and qrels for 24 qids.
Evaluating 24 QA items that have qrels.

=== Strategy A — Confidence-Based Routing — quantitative summary ===
        method  queries_evaluated      MRR  Precision@1  Recall@1  Precision@3  Recall@3  Precision@5  Recall@5  Precision@10  Recall@10
Confidence (A)                 24 0.364583     0.291667  0.023399     0.152778  0.025065     0.141667  0.029486        0.1125   0.033084


### Step 6 — Efficiency evaluation

**What to implement in the next code cell:**
Time each query and aggregate efficiency stats, paying special attention to **gating** and **retry** behavior since those are Confidence routing's main efficiency levers.

**Report:**
- average latency
- median latency
- P95 latency
- **retry count** — how often the critic triggered a re-retrieval
- **gating count** — how often each retriever was skipped (per `query_type`)
- optional cost proxy (number of agent calls per query)

**Reason:**
- Step 3 explicitly asks for system efficiency evidence.
- Gating is Confidence routing's *speed advantage*. If our gating count is always 0, the strategy reduces to plain Voting and we lose the latency story in the report.

**Learning point:** Save raw per-query latencies (not just the mean). Box plots in Step 8 will need them.

In [29]:
# === Step 6 — Efficiency evaluation ===
# Confidence routing's main efficiency story = gating skips retrievers,
# and retry adds work only when the critic flags an ungrounded answer.
# We surface BOTH so the trade-off is visible in the report.

import numpy as np

latencies = np.array(list(latencies_confidence.values()), dtype=float)

n_total   = len(latencies)
n_retry   = sum(1 for t in traces_confidence.values() if t['retry_triggered'])

# Gating count per (query_type, retriever) — i.e. how often each retriever
# was skipped for each query type. If everything is 0, gating never fires
# and Confidence reduces to weighted Voting.
gating_per_type = defaultdict(lambda: defaultdict(int))
type_counts     = defaultdict(int)
retrieval_error_counts = defaultdict(int)
zero_result_counts = defaultdict(int)
for t in traces_confidence.values():
    type_counts[t['query_type']] += 1
    for r in t['gated_out']:
        gating_per_type[t['query_type']][r] += 1
    for r in t.get('retrieval_errors', {}):
        retrieval_error_counts[r] += 1
    for r in t.get('zero_result_retrievers', []):
        zero_result_counts[r] += 1

efficiency_summary = pd.Series({
    'queries_total'   : n_total,
    'avg_latency_s'   : float(latencies.mean())            if n_total else 0.0,
    'median_latency_s': float(np.median(latencies))        if n_total else 0.0,
    'p95_latency_s'   : float(np.percentile(latencies, 95)) if n_total else 0.0,
    'min_latency_s'   : float(latencies.min())             if n_total else 0.0,
    'max_latency_s'   : float(latencies.max())             if n_total else 0.0,
    'retry_count'     : n_retry,
    'retry_rate'      : n_retry / max(n_total, 1),
})
print('=== Efficiency summary ===')
print(efficiency_summary.to_string())

print('\n=== Gating count per query_type and retriever ===')
print('(reads as: of the N queries of this type, how often was each retriever skipped)')
for qt, total in type_counts.items():
    skipped = dict(gating_per_type.get(qt, {}))
    print(f'  {qt:18s} (n={total}): {skipped if skipped else "no gating"}')

print('\n=== Retrieval health check ===')
print('retrieval_errors:', dict(retrieval_error_counts) if retrieval_error_counts else 'none')
print('zero_result_retrievers:', dict(zero_result_counts) if zero_result_counts else 'none')
if retrieval_error_counts or zero_result_counts:
    print('Warning: investigate traces_confidence before reporting final metrics.')

=== Efficiency summary ===
queries_total       24.000000
avg_latency_s        0.222138
median_latency_s     0.208550
p95_latency_s        0.319750
min_latency_s        0.169100
max_latency_s        0.348900
retry_count          0.000000
retry_rate           0.000000

=== Gating count per query_type and retriever ===
(reads as: of the N queries of this type, how often was each retriever skipped)
  entity_temporal    (n=2): {'bm25': 2}
  entity             (n=4): no gating
  keyword            (n=6): {'graph': 6}
  semantic           (n=8): no gating
  mixed              (n=4): no gating

=== Retrieval health check ===
retrieval_errors: none
zero_result_retrievers: none


### Step 7 — Qualitative analysis block

**What to implement in the next code cell:**
Three sub-analyses, each just a few cells of code:

1. **Explainability examples** — pick 3–5 queries (mix of types) and print the full trace + final answer for each. This becomes Stretch Goal A in the plan with almost zero extra code.

2. **Agent complementarity** — for each query, compute pairwise overlap (Jaccard) between BM25 / Dense / Graph top-k document sets. Average across queries. High overlap = retrievers are redundant. Low overlap = each adds unique signal.

3. **Failure analysis** — list queries with `MRR == 0` or `P@1 == 0` and categorize:
   - language mismatch (German query, English-leaning retrieval)
   - query type misclassification (e.g. semantic question routed to keyword preset)
   - missing-from-corpus (no relevant doc exists at all)
   - gating mistake (the right retriever was gated out)

**Reason:**
- Required for the qualitative deliverable in Step 3b.
- Categorized failures **directly inform tuning** of `WEIGHT_PRESETS` and `GATE_THRESHOLD` — close the loop.

**Learning point:** Keep failure categories *mutually exclusive and exhaustive* — otherwise the percentages won't add up to 100% and the report table looks sloppy.

In [30]:
# === Step 7 — Qualitative analysis ===
# Three sub-analyses, all powered by traces_confidence + perq_conf from Step 5/6:
#   (a) Explainability — show 3 full traces (covers Stretch Goal A in the plan).
#   (b) Agent complementarity — Jaccard overlap between BM25/Dense/Graph top-k.
#   (c) Failure analysis — categorise queries with MRR == 0.

# ---------- (a) Explainability examples ----------
print('=' * 80)
print('(a) Explainability examples — 3 traces in full')
print('=' * 80)
sample_qids = list(traces_confidence.keys())[:3]
for qid in sample_qids:
    t = traces_confidence[qid]
    print(f"\n[QID {qid}] type={t['query_type']} | weights={t['weights']}")
    print(f"          gated_out={t['gated_out']} | retry={t['retry_triggered']} "
          f"| critic_ok={t['critic_ok']}")
    print(f"  query    : {t['query']}")
    print(f"  evidence : {t['evidence_ids'][:3]}")
    print(f"  counts   : {t['retriever_counts']}")
    print(f"  errors   : {t.get('retrieval_errors', {}) or 'none'}")
    print(f"  zero     : {t.get('zero_result_retrievers', []) or 'none'}")
    print(f"  latency  : {t['latency_s']}s")

# ---------- (b) Agent complementarity ----------
# Run BM25 / Dense / Graph independently on a small evaluated sample and measure
# Jaccard overlap of their top-10 sets. Low overlap => each retriever adds
# unique signal => fusion is worth the cost. Retrieval failures are printed
# instead of silently converted into empty sets.
print('\n' + '=' * 80)
print('(b) Agent complementarity — mean Jaccard overlap on top-10 (sample of 5 evaluated queries)')
print('=' * 80)
overlaps = {('bm25', 'dense'): [], ('bm25', 'graph'): [], ('dense', 'graph'): []}
complementarity_errors = defaultdict(int)
for item in eval_qa_data[:5]:
    q = item.get('question') or item.get('query')
    state = AgentState(query=q)
    state = orchestrator.query_agent.run(state)
    sets = {}
    for name, agent in orchestrator.retrievers.items():
        try:
            agent.run(state, top_k=10)
            sets[name] = {_uid(d) for d in state.retrieval_by_agent.get(name, [])
                          if _uid(d) is not None}
            if not sets[name]:
                complementarity_errors[f'{name}: zero docs'] += 1
        except Exception as e:
            sets[name] = set()
            complementarity_errors[f'{name}: {type(e).__name__}'] += 1
    for (a, b) in overlaps:
        union = sets[a] | sets[b]
        inter = sets[a] & sets[b]
        overlaps[(a, b)].append(len(inter) / max(len(union), 1))

for pair, vals in overlaps.items():
    mean_j = sum(vals) / max(len(vals), 1)
    print(f'  {pair[0]:5s} <-> {pair[1]:5s} : mean Jaccard = {mean_j:.3f}')
if complementarity_errors:
    print('Complementarity warnings:', dict(complementarity_errors))
print('\nReading guide: low Jaccard (<0.2) means retrievers see different docs.')
print('That is GOOD for fusion only if each retriever returned real documents.')

# ---------- (c) Failure analysis ----------
print('\n' + '=' * 80)
print('(c) Failure analysis — queries with MRR == 0')
print('=' * 80)
failures = perq_conf[perq_conf['MRR'] == 0].copy()
print(f'  {len(failures)} of {len(perq_conf)} queries failed (MRR == 0).')

# Categorise as a list of CONTRIBUTING FLAGS (not mutually exclusive root causes).
# Why flags instead of single buckets:
#   * Gating a low-weight retriever can be the CORRECT design choice; calling
#     it a "mistake" whenever it appears would be misleading in the report.
#   * Multiple contributing factors often coexist (e.g. German query AND retry
#     failed). Flag-style labels preserve that information.
def _categorise(qid):
    t = traces_confidence.get(qid, {})
    q = (t.get('query') or '').lower()
    # NOTE: this DE check only catches our ~10 hardcoded keywords. Queries like
    # "Was ist die ETH?" with no mapped keyword will fall through to
    # missing_or_low_signal — call this out when interpreting the table.
    is_de = any(re.search(r'\b' + de + r'\b', q) for de in DE_EN_TERM_MAP)
    flags = []
    if t.get('retrieval_errors'):
        flags.append('retrieval_error')
    if t.get('zero_result_retrievers'):
        flags.append('zero_result_retriever')
    if is_de and t.get('bilingual_variants', 1) == 1:
        flags.append('possible_language_gap')
    if t.get('gated_out'):
        flags.append('gating_applied')        # neutral observation, not a verdict
    if t.get('retry_triggered') and not t.get('critic_ok'):
        flags.append('retry_failed')
    if not flags:
        flags.append('missing_or_low_signal')
    return ' | '.join(flags)

failures['flags'] = failures['qid'].apply(_categorise)
print('\nFailure flag distribution (queries can carry multiple flags):')
print(failures['flags'].value_counts().to_string())
print('\nFirst 10 failed queries:')
print(failures[['qid', 'flags', 'Precision@5', 'Recall@5']].head(10).to_string(index=False))

(a) Explainability examples — 3 traces in full

[QID 2] type=entity_temporal | weights={'dense': 1.1, 'graph': 1.4}
          gated_out=['bm25'] | retry=False | critic_ok=True
  query    : who were the rectors of eth between 2017 and 2022?
  evidence : ['73afc89edd471ff98176f33babf28b62dccf4ac6_fixed_2', '00859327fafc62821b53b9ad083e2a244c1b4470_fixed_3', '20aed32900c864b6d99d046fa3706c8dec5194d6_fixed_0']
  counts   : {'bm25': 0, 'dense': 50, 'graph': 50}
  errors   : none
  zero     : none
  latency  : 0.2122s

[QID 3] type=entity | weights={'bm25': 0.9, 'dense': 1.1, 'graph': 1.3}
          gated_out=[] | retry=False | critic_ok=True
  query    : who at eth received erc grants?
  evidence : ['455c5e6fa03aaf70b745bcec98f92048a0fdd3d1_fixed_0', '327dfd590409db9eb6c600a5e3b71792d37d8075_fixed_1', 'b391e9189684d3b2a19d0da5edf7de0f3e5d7cfa_fixed_0']
  counts   : {'bm25': 50, 'dense': 50, 'graph': 50}
  errors   : none
  zero     : none
  latency  : 0.3036s

[QID 4] type=keyword | weights

## Strategy B: Sequential Waterfall Orchestration

This strategy attempts to solve queries using the most efficient resources first. It follows a 'Waterfall' logic:
1. **Tier 1 (Lexical):** BM25 only. If the Critic is satisfied, return immediately.
2. **Tier 2 (Semantic):** Dense + BM25 Fusion. If the Critic is satisfied, return.
3. **Tier 3 (Full):** BM25 + Dense + GraphRAG Fusion (The high-latency fallback).

In [ ]:
class WaterfallOrchestrator(ConfidenceOrchestrator):
    """Sequential Waterfall Strategy: minimizes latency by trying cheaper retrievers first."""

    def run(self, query: str, retrieve_k: int = RETRIEVE_K, top_k: int = TOP_K):
        t0 = time.time()
        state = AgentState(query=query)
        state.retrieval_errors = {}
        state.zero_result_retrievers = []
        state = self.query_agent.run(state)
        original_norm = state.normalized_query

        tiers = [
            (['bm25'], {'bm25': 1.0}),
            (['bm25', 'dense'], {'bm25': 1.0, 'dense': 1.2}),
            (['bm25', 'dense', 'graph'], {'bm25': 1.0, 'dense': 1.2, 'graph': 1.4}),
        ]

        trace = {
            'query': query,
            'query_type': state.query_type,
            'tiers_attempted': 0,
            'tier_results': [],
        }

        for i, (active_names, weights) in enumerate(tiers, 1):
            trace['tiers_attempted'] = i
            state = self._retrieve(state, active_names, retrieve_k, [original_norm])
            state.normalized_query = original_norm
            state.query_hints = {**state.query_hints, **weights}

            state = self.fusion_agent.run(state, top_k=retrieve_k)
            state = self.reranker_agent.run(state, top_k=top_k)
            state = self.answer_agent.run(state)
            state = self.critic_agent.run(state)

            trace['tier_results'].append({
                'tier': i,
                'active': list(active_names),
                'weights': dict(weights),
                'critic_ok': state.critic_ok,
                'retriever_counts': {n: len(state.retrieval_by_agent.get(n, []))
                                     for n in self.retrievers},
                'retrieval_errors': dict(getattr(state, 'retrieval_errors', {})),
                'zero_result_retrievers': list(getattr(state, 'zero_result_retrievers', [])),
            })

            if state.critic_ok:
                break

        if not state.critic_ok:
            year_hint = state.query_hints.get('_year')
            year_str = f' from {year_hint}' if year_hint else ''
            state.final_answer = (
                f'The available corpus does not contain sufficient evidence to '
                f'reliably answer this query{year_str}. Retrieved context covers '
                f'related topics but does not directly address the question.'
            )

        latency = time.time() - t0
        trace.update({
            'latency_s': round(latency, 4),
            'final_tier': i,
            'critic_ok': state.critic_ok,
            'critic_feedback': state.critic_feedback,
            'retriever_counts': {n: len(state.retrieval_by_agent.get(n, []))
                                 for n in self.retrievers},
            'retrieval_errors': dict(getattr(state, 'retrieval_errors', {})),
            'zero_result_retrievers': list(getattr(state, 'zero_result_retrievers', [])),
            'evidence_ids': state.evidence_ids,
        })

        top_docs = (state.reranked_docs or state.fused_docs)[:top_k]
        return state.final_answer, top_docs, trace

waterfall_orchestrator = WaterfallOrchestrator()
print('Waterfall orchestrator ready.')

In [ ]:
# Comparative Evaluation: Strategy A (Confidence) vs Strategy B (Waterfall)
sample_q = "Who was the president of ETH in 2003?"

print(f"Testing Query: {sample_q}\n")

_, _, trace_a = orchestrator.run(sample_q)
print(f"[Confidence A] Latency: {trace_a['latency_s']}s | Gated: {trace_a['gated_out']}")

_, _, trace_b = waterfall_orchestrator.run(sample_q)
print(f"[Waterfall B] Latency: {trace_b['latency_s']}s | Tiers run: {trace_b['tiers_attempted']}")

### Strategy B Verification: Tier Escalation Test
This test ensures that the Waterfall strategy handles both success (stopping early) and failure (escalating to next tiers) as expected based on the Critic's feedback.

In [ ]:
import pandas as pd

def verify_waterfall_tiers(queries):
    results = []
    for q in queries:
        _, _, trace = waterfall_orchestrator.run(q)
        results.append({
            'Query': q,
            'Tiers Attempted': trace['tiers_attempted'],
            'Final Tier Status': 'Critic OK' if trace['critic_ok'] else 'Exhausted Tiers',
            'Latency (s)': trace['latency_s']
        })
    return pd.DataFrame(results)

# Test set: one likely Tier 1 (lexical) and one likely Tier 3 (complex/semantic)
verification_queries = [
    "Who was the president of ETH in 2003?",
    "Explain the cross-disciplinary impact of sustainability research on ETH industry partnerships between 2010 and 2020."
]

display(verify_waterfall_tiers(verification_queries))

### Strategy B Integrity Check
To verify that the Waterfall isn't just stopping at Tier 1 by accident, we will test it with a query that is unlikely to be solved by simple lexical BM25 (Tier 1) but should require Semantic/Dense (Tier 2/3). This confirms the 'Waterfall' actually flows when needed.

In [ ]:
import time

# A complex query that usually requires semantic understanding beyond keywords
integrity_query = "How does the ETH executive board balance academic freedom with industrial research funding constraints?"

print(f"Testing Tier Escalation for: {integrity_query}")
_, _, trace = waterfall_orchestrator.run(integrity_query)

print(f"\n--- Results ---")
print(f"Tiers attempted: {trace['tiers_attempted']}")
print(f"Final Tier: {trace['final_tier']}")
print(f"Critic OK: {trace['critic_ok']}")
print(f"Latency: {trace['latency_s']}s")

for res in trace['tier_results']:
    print(f"Tier {res['tier']} ({', '.join(res['active'])}): Critic OK = {res['critic_ok']}")

### Step 10 — Quantitative & Efficiency Evaluation for Strategy B (Waterfall)

We will now run the benchmark against the 24 project questions using the `waterfall_orchestrator`. We will collect both IR metrics and efficiency data to compare against Strategy A.

In [ ]:
run_waterfall = {}
traces_waterfall = {}
latencies_waterfall = {}

print(f'Evaluating {len(eval_qa_data)} QA items with Strategy B Waterfall...')
for item in eval_qa_data:
    qid = str(item['id'])
    query = item.get('question') or item.get('query')
    answer, docs, trace = waterfall_orchestrator.run(query,
                                                     retrieve_k=RETRIEVE_K,
                                                     top_k=max(K_VALUES))
    run_waterfall[qid] = [_uid(d) for d in docs if _uid(d) is not None]
    traces_waterfall[qid] = trace
    latencies_waterfall[qid] = trace['latency_s']

summary_waterfall, perq_waterfall = evaluate_run('Waterfall (B)', run_waterfall, qrels)
waterfall_latencies = np.array(list(latencies_waterfall.values()), dtype=float)
tiers_attempted = [t['tiers_attempted'] for t in traces_waterfall.values()]
waterfall_error_counts = defaultdict(int)
waterfall_zero_counts = defaultdict(int)
for trace in traces_waterfall.values():
    for name in trace.get('retrieval_errors', {}):
        waterfall_error_counts[name] += 1
    for name in trace.get('zero_result_retrievers', []):
        waterfall_zero_counts[name] += 1

waterfall_eff_summary = pd.Series({
    'queries_total': len(run_waterfall),
    'avg_latency_s': float(waterfall_latencies.mean()) if len(waterfall_latencies) else 0.0,
    'p95_latency_s': float(np.percentile(waterfall_latencies, 95)) if len(waterfall_latencies) else 0.0,
    'avg_tiers_run': float(np.mean(tiers_attempted)) if tiers_attempted else 0.0,
    'tier_1_stop_rate': sum(1 for t in tiers_attempted if t == 1) / max(len(tiers_attempted), 1),
    'critic_fail_count': sum(1 for t in traces_waterfall.values() if not t['critic_ok']),
})

print('\n=== Strategy B — Waterfall Orchestration — quantitative summary ===')
display(pd.DataFrame([summary_waterfall])[ordered_cols])

print('\n=== Strategy B — Efficiency Summary ===')
display(waterfall_eff_summary)

print('\n=== Strategy B — Retrieval Health Check ===')
print('retrieval_errors:', dict(waterfall_error_counts) if waterfall_error_counts else 'none')
print('zero_result_retrievers:', dict(waterfall_zero_counts) if waterfall_zero_counts else 'none')
if waterfall_error_counts or waterfall_zero_counts:
    print('Warning: investigate traces_waterfall before reporting final Strategy B metrics.')

## Strategy C: Equal-Weight Voting Orchestration

This section separates the previous voting baseline into its own strategy block.

**Strategy C logic:**
- Run BM25, Dense, and Graph for every query.
- Use equal fusion weights: `bm25 = dense = graph = 1.0`.
- Disable gating.
- Disable critic-driven retry.
- Use the same `eval_qa_data`, qrels, `RETRIEVE_K`, `TOP_K`, and `K_VALUES` as Strategy A and Strategy B.

This makes Strategy C a fair comparison point:
- **Strategy A** = Confidence routing with query-type weights, gating, and optional retry.
- **Strategy B** = Sequential Waterfall with tier escalation.
- **Strategy C** = Parallel equal-weight voting with no routing shortcuts.

In [ ]:
# === Strategy C — Equal-Weight Voting Orchestration ===
# This is a standalone strategy, not just a baseline row.
# It uses the same wrapper as Strategy A for fairness, but disables:
#   - query-type weighting differences
#   - gating
#   - critic-driven retry

import time
import matplotlib.pyplot as plt

voting_weight_presets = {
    key: {'bm25': 1.0, 'dense': 1.0, 'graph': 1.0}
    for key in WEIGHT_PRESETS
}

voting_orchestrator = ConfidenceOrchestrator(
    weight_presets=voting_weight_presets,
    gate_threshold=0.0,
    use_gating=False,
    use_retry=False,
    use_bilingual=USE_BILINGUAL_QUERY,
)

run_voting       = {}
traces_voting    = {}
latencies_voting = {}

print(f'Evaluating {len(eval_qa_data)} QA items with Strategy C Voting...')
for item in eval_qa_data:
    qid = str(item['id'])
    q   = item.get('question') or item.get('query')
    t0  = time.time()
    answer, docs, trace = voting_orchestrator.run(q,
                                                  retrieve_k=RETRIEVE_K,
                                                  top_k=max(K_VALUES))
    latencies_voting[qid] = time.time() - t0
    run_voting[qid] = [_uid(d) for d in docs if _uid(d) is not None]
    traces_voting[qid] = trace

summary_vote, perq_vote = evaluate_run('Voting (C)', run_voting, qrels)

vote_latency_values = [latencies_voting[qid] for qid in sorted(latencies_voting)]
vote_error_counts = defaultdict(int)
vote_zero_counts = defaultdict(int)
for trace in traces_voting.values():
    for name in trace.get('retrieval_errors', {}):
        vote_error_counts[name] += 1
    for name in trace.get('zero_result_retrievers', []):
        vote_zero_counts[name] += 1

voting_eff_summary = pd.Series({
    'queries_total': len(run_voting),
    'avg_latency_s': float(np.mean(vote_latency_values)) if vote_latency_values else 0.0,
    'p95_latency_s': float(np.percentile(vote_latency_values, 95)) if vote_latency_values else 0.0,
    'retry_count': sum(1 for t in traces_voting.values() if t['retry_triggered']),
})

print('\n=== Strategy C — Equal-Weight Voting — quantitative summary ===')
display(pd.DataFrame([summary_vote])[ordered_cols])

print('\n=== Strategy C — Efficiency Summary ===')
display(voting_eff_summary)

print('\n=== Strategy C — Retrieval Health Check ===')
print('retrieval_errors:', dict(vote_error_counts) if vote_error_counts else 'none')
print('zero_result_retrievers:', dict(vote_zero_counts) if vote_zero_counts else 'none')

### Final comparative analysis — Strategy A vs Strategy B vs Strategy C

This section compares all implemented orchestration strategies on the same evaluated query set.

- **Strategy A:** Confidence routing.
- **Strategy B:** Sequential Waterfall.
- **Strategy C:** Equal-weight Voting.

The table uses the same metric columns as the baseline report. The latency plot uses raw per-query latency values from each strategy.

In [ ]:
final_compare_df = pd.DataFrame([summary_conf, summary_waterfall, summary_vote])[ordered_cols]
print('=== Final Strategy Comparison ===')
display(final_compare_df)

latency_series = [
    [latencies_confidence[qid] for qid in sorted(latencies_confidence)],
    [latencies_waterfall[qid] for qid in sorted(latencies_waterfall)],
    [latencies_voting[qid] for qid in sorted(latencies_voting)],
]
latency_labels = ['Confidence (A)', 'Waterfall (B)', 'Voting (C)']

plt.figure(figsize=(10, 5))
plt.boxplot(latency_series)
plt.xticks([1, 2, 3], latency_labels)
plt.title('End-to-End Latency Comparison')
plt.ylabel('Seconds')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.show()

strategy_efficiency_df = pd.DataFrame([
    {
        'method': 'Confidence (A)',
        'queries_total': len(latencies_confidence),
        'avg_latency_s': float(np.mean(latency_series[0])) if latency_series[0] else 0.0,
        'p95_latency_s': float(np.percentile(latency_series[0], 95)) if latency_series[0] else 0.0,
        'retry_count': sum(1 for t in traces_confidence.values() if t['retry_triggered']),
    },
    {
        'method': 'Waterfall (B)',
        'queries_total': len(latencies_waterfall),
        'avg_latency_s': float(np.mean(latency_series[1])) if latency_series[1] else 0.0,
        'p95_latency_s': float(np.percentile(latency_series[1], 95)) if latency_series[1] else 0.0,
        'retry_count': 0,
        'avg_tiers_run': waterfall_eff_summary.get('avg_tiers_run', None),
        'critic_fail_count': waterfall_eff_summary.get('critic_fail_count', None),
    },
    {
        'method': 'Voting (C)',
        'queries_total': len(latencies_voting),
        'avg_latency_s': float(np.mean(latency_series[2])) if latency_series[2] else 0.0,
        'p95_latency_s': float(np.percentile(latency_series[2], 95)) if latency_series[2] else 0.0,
        'retry_count': sum(1 for t in traces_voting.values() if t['retry_triggered']),
    },
])

print('\n=== Strategy Efficiency Comparison ===')
display(strategy_efficiency_df)

RESULTS_DIR = PROJECT_ROOT / 'results' / f'strategy_comparison_{EVAL_SCOPE}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
final_compare_df.to_csv(RESULTS_DIR / 'summary_strategies_A_B_C.csv', index=False)
strategy_efficiency_df.to_csv(RESULTS_DIR / 'efficiency_strategies_A_B_C.csv', index=False)
perq_conf.to_csv(RESULTS_DIR / 'per_query_confidence_A.csv', index=False)
perq_waterfall.to_csv(RESULTS_DIR / 'per_query_waterfall_B.csv', index=False)
perq_vote.to_csv(RESULTS_DIR / 'per_query_voting_C.csv', index=False)

with open(RESULTS_DIR / 'traces_confidence_A.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'evidence_ids'}
               for k, v in traces_confidence.items()}, f, indent=2, default=str)
with open(RESULTS_DIR / 'traces_waterfall_B.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'evidence_ids'}
               for k, v in traces_waterfall.items()}, f, indent=2, default=str)
with open(RESULTS_DIR / 'traces_voting_C.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'evidence_ids'}
               for k, v in traces_voting.items()}, f, indent=2, default=str)

print(f'Saved Strategy A/B/C artifacts to {RESULTS_DIR.resolve()}')

try:
    from scipy.stats import ttest_rel
    perq_frames = {
        'Confidence (A)': perq_conf.set_index('qid'),
        'Waterfall (B)': perq_waterfall.set_index('qid'),
        'Voting (C)': perq_vote.set_index('qid'),
    }
    pairs = [('Confidence (A)', 'Waterfall (B)'),
             ('Confidence (A)', 'Voting (C)'),
             ('Waterfall (B)', 'Voting (C)')]
    print('\n=== Paired t-tests on per-query MRR ===')
    for left, right in pairs:
        common = sorted(set(perq_frames[left].index) & set(perq_frames[right].index))
        a = perq_frames[left].loc[common, 'MRR'].values
        b = perq_frames[right].loc[common, 'MRR'].values
        if len(common) == 0:
            print(f'{left} vs {right}: skipped, no common qids')
        elif np.allclose(a, b):
            print(f'{left} vs {right}: skipped, per-query MRR values are identical')
        else:
            t_stat, p_val = ttest_rel(a, b)
            print(f'{left} vs {right}: t={t_stat:.3f}, p={p_val:.3f}')
except Exception as e:
    print(f'Paired t-tests skipped: {e}')

## Report-aligned evaluation tables

This final section follows the same table structure as `baseline_repro_report.md`, so the Strategy A/B/C notebook outputs can be compared directly with the reproduced baseline results.

It creates:
- **Baseline evaluation table** for full corpus, using the same columns as the report.
- **Orchestration evaluation table** for Strategy A/B/C, using the same columns and order.
- **Full-corpus combined table** for baseline methods and orchestration methods together.
- **Subsample vs full-corpus comparison table** with the compact `Scope`, `Method`, `MRR`, `P@k`, and `R@k` layout from the report.

Important interpretation note: the baseline report's text describes Waterfall differently from the current notebook's Strategy B. In this notebook, **Strategy B Waterfall** escalates by critic feedback: BM25 → BM25+Dense → BM25+Dense+Graph.

In [ ]:
# === Report-aligned evaluation tables ===
# These tables intentionally mirror baseline_repro_report.md.
# Run this after Strategy A, Strategy B, Strategy C, and the final A/B/C comparison cell.

REPORT_RESULTS_DIR = PROJECT_ROOT / 'results' / f'report_aligned_strategy_comparison_{EVAL_SCOPE}'
REPORT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REPORT_ORDERED_COLS = ['method', 'queries_evaluated', 'MRR']
for k in K_VALUES:
    REPORT_ORDERED_COLS += [f'Precision@{k}', f'Recall@{k}']

REPORT_COMPACT_COLS = ['Scope', 'Method', 'MRR']
for k in K_VALUES:
    REPORT_COMPACT_COLS.append(f'P@{k}')
for k in K_VALUES:
    REPORT_COMPACT_COLS.append(f'R@{k}')

BASELINE_METHOD_ORDER = ['GraphRAG', 'ReRank', 'Hybrid', 'Dense', 'BM25']
BASELINE_SCOPE_METHOD_ORDER = ['BM25', 'Dense', 'GraphRAG', 'Hybrid', 'ReRank']
ORCHESTRATION_METHOD_ORDER = ['Confidence', 'Waterfall', 'Voting']

FULL_BASELINE_VALUES = {
    'GraphRAG': {'queries_evaluated': 24, 'MRR': 0.232573, 'Precision@1': 0.083333, 'Recall@1': 0.000687, 'Precision@3': 0.097222, 'Recall@3': 0.003166, 'Precision@5': 0.116667, 'Recall@5': 0.005892, 'Precision@10': 0.116667, 'Recall@10': 0.052857},
    'ReRank': {'queries_evaluated': 24, 'MRR': 0.222952, 'Precision@1': 0.041667, 'Recall@1': 0.000196, 'Precision@3': 0.138889, 'Recall@3': 0.004480, 'Precision@5': 0.125000, 'Recall@5': 0.006323, 'Precision@10': 0.100000, 'Recall@10': 0.010619},
    'Hybrid': {'queries_evaluated': 24, 'MRR': 0.202154, 'Precision@1': 0.000000, 'Recall@1': 0.000000, 'Precision@3': 0.097222, 'Recall@3': 0.003699, 'Precision@5': 0.125000, 'Recall@5': 0.028352, 'Precision@10': 0.095833, 'Recall@10': 0.031149},
    'Dense': {'queries_evaluated': 24, 'MRR': 0.165525, 'Precision@1': 0.041667, 'Recall@1': 0.000147, 'Precision@3': 0.069444, 'Recall@3': 0.008953, 'Precision@5': 0.058333, 'Recall@5': 0.010095, 'Precision@10': 0.066667, 'Recall@10': 0.034097},
    'BM25': {'queries_evaluated': 24, 'MRR': 0.151296, 'Precision@1': 0.041667, 'Recall@1': 0.001016, 'Precision@3': 0.055556, 'Recall@3': 0.001681, 'Precision@5': 0.091667, 'Recall@5': 0.005506, 'Precision@10': 0.091667, 'Recall@10': 0.011165},
}

SUBSAMPLE_BASELINE_VALUES = {
    'BM25': {'queries_evaluated': 24, 'MRR': 0.396272, 'Precision@1': 0.166667, 'Precision@3': 0.305556, 'Precision@5': 0.316667, 'Precision@10': 0.295833, 'Recall@1': 0.002081, 'Recall@3': 0.010785, 'Recall@5': 0.017426, 'Recall@10': 0.037034},
    'Dense': {'queries_evaluated': 24, 'MRR': 0.540476, 'Precision@1': 0.375000, 'Precision@3': 0.402778, 'Precision@5': 0.350000, 'Precision@10': 0.304167, 'Recall@1': 0.031450, 'Recall@3': 0.064326, 'Recall@5': 0.073926, 'Recall@10': 0.093394},
    'GraphRAG': {'queries_evaluated': 24, 'MRR': 0.579613, 'Precision@1': 0.458333, 'Precision@3': 0.444444, 'Precision@5': 0.366667, 'Precision@10': 0.337500, 'Recall@1': 0.029346, 'Recall@3': 0.059344, 'Recall@5': 0.063712, 'Recall@10': 0.080849},
    'Hybrid': {'queries_evaluated': 24, 'MRR': 0.462108, 'Precision@1': 0.250000, 'Precision@3': 0.347222, 'Precision@5': 0.316667, 'Precision@10': 0.316667, 'Recall@1': 0.004266, 'Recall@3': 0.062629, 'Recall@5': 0.076968, 'Recall@10': 0.097059},
    'ReRank': {'queries_evaluated': 24, 'MRR': 0.442001, 'Precision@1': 0.291667, 'Precision@3': 0.250000, 'Precision@5': 0.233333, 'Precision@10': 0.245833, 'Recall@1': 0.002821, 'Recall@3': 0.006986, 'Recall@5': 0.012018, 'Recall@10': 0.029840},
}

def _normalise_report_columns(df):
    out = df.copy()
    rename_map = {}
    for col in out.columns:
        lower = str(col).lower()
        if lower == 'method':
            rename_map[col] = 'method'
        elif lower == 'queries_evaluated':
            rename_map[col] = 'queries_evaluated'
    out = out.rename(columns=rename_map)
    return out

def _summary_rows_from_values(values, method_order):
    rows = []
    for method in method_order:
        row = {'method': method, 'queries_evaluated': values[method].get('queries_evaluated', 24)}
        row.update(values[method])
        rows.append(row)
    return pd.DataFrame(rows)[REPORT_ORDERED_COLS]

def _read_baseline_summary(scope):
    path = PROJECT_ROOT / 'results' / f'baseline_repro_colab_{scope}' / 'metrics_summary.csv'
    if not path.exists():
        return None
    df = _normalise_report_columns(pd.read_csv(path))
    return df[REPORT_ORDERED_COLS]

def _clean_orchestration_method_name(name):
    name = str(name)
    if 'Confidence' in name:
        return 'Confidence'
    if 'Waterfall' in name:
        return 'Waterfall'
    if 'Voting' in name:
        return 'Voting'
    return name

def _sort_methods(df, order):
    out = df.copy()
    out['_order'] = out['method'].map({name: i for i, name in enumerate(order)})
    out['_order'] = out['_order'].fillna(999)
    return out.sort_values(['_order', 'method']).drop(columns=['_order']).reset_index(drop=True)

def _to_compact_scope_table(df, scope):
    rows = []
    for _, row in df.iterrows():
        out = {
            'Scope': scope,
            'Method': row['method'],
            'MRR': row['MRR'],
        }
        for k in K_VALUES:
            out[f'P@{k}'] = row[f'Precision@{k}']
        for k in K_VALUES:
            out[f'R@{k}'] = row[f'Recall@{k}']
        rows.append(out)
    return pd.DataFrame(rows)[REPORT_COMPACT_COLS]

def _format_report_numbers(df):
    out = df.copy()
    for col in out.columns:
        if col in {'method', 'Method', 'Scope', 'system_group'}:
            continue
        if col in {'queries_evaluated'}:
            out[col] = out[col].astype(int)
        else:
            out[col] = out[col].astype(float).round(6)
    return out

def _load_orchestration_scope_summary(scope):
    path = PROJECT_ROOT / 'results' / f'strategy_comparison_{scope}' / 'summary_strategies_A_B_C.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df['method'] = df['method'].apply(_clean_orchestration_method_name)
    return _sort_methods(df[REPORT_ORDERED_COLS], ORCHESTRATION_METHOD_ORDER)

baseline_full_report_df = _read_baseline_summary('full_corpus')
if baseline_full_report_df is None:
    baseline_full_report_df = _summary_rows_from_values(FULL_BASELINE_VALUES, BASELINE_METHOD_ORDER)
else:
    baseline_full_report_df = _sort_methods(baseline_full_report_df, BASELINE_METHOD_ORDER)

baseline_subsample_report_df = _read_baseline_summary('subsample')
if baseline_subsample_report_df is None:
    baseline_subsample_report_df = _summary_rows_from_values(SUBSAMPLE_BASELINE_VALUES, BASELINE_SCOPE_METHOD_ORDER)
else:
    baseline_subsample_report_df = _sort_methods(baseline_subsample_report_df, BASELINE_SCOPE_METHOD_ORDER)

orchestration_report_df = final_compare_df.copy()
orchestration_report_df['method'] = orchestration_report_df['method'].apply(_clean_orchestration_method_name)
orchestration_report_df = _sort_methods(orchestration_report_df[REPORT_ORDERED_COLS], ORCHESTRATION_METHOD_ORDER)

full_system_report_df = pd.concat([baseline_full_report_df, orchestration_report_df], ignore_index=True)
full_system_report_df['system_group'] = ['Baseline'] * len(baseline_full_report_df) + ['Orchestration'] * len(orchestration_report_df)
full_system_report_df = full_system_report_df[['system_group'] + REPORT_ORDERED_COLS]

baseline_scope_report_df = pd.concat([
    _to_compact_scope_table(_sort_methods(baseline_full_report_df, BASELINE_SCOPE_METHOD_ORDER), 'full_corpus'),
    _to_compact_scope_table(_sort_methods(baseline_subsample_report_df, BASELINE_SCOPE_METHOD_ORDER), 'subsample'),
], ignore_index=True).sort_values(['Method', 'Scope']).reset_index(drop=True)

orchestration_scope_frames = []
for scope in ['full_corpus', 'subsample']:
    scope_df = _load_orchestration_scope_summary(scope)
    if scope_df is not None:
        orchestration_scope_frames.append(_to_compact_scope_table(scope_df, scope))

orchestration_scope_report_df = pd.concat(orchestration_scope_frames, ignore_index=True) if orchestration_scope_frames else _to_compact_scope_table(orchestration_report_df, EVAL_SCOPE)

print('=== 2.1 Baseline Evaluation (Full Corpus) — report structure ===')
display(_format_report_numbers(baseline_full_report_df))

print('\n=== 2.2 Orchestration Evaluation ({}) — report structure ==='.format(EVAL_SCOPE))
display(_format_report_numbers(orchestration_report_df))

print('\n=== Baseline + Orchestration Comparison ({}) ==='.format(EVAL_SCOPE))
display(_format_report_numbers(full_system_report_df))

print('\n=== 2.3 Baseline Subsample vs Full Corpus Comparison — report structure ===')
display(_format_report_numbers(baseline_scope_report_df))

print('\n=== Notebook Orchestration Scope Comparison ===')
display(_format_report_numbers(orchestration_scope_report_df))
if set(orchestration_scope_report_df['Scope']) != {'full_corpus', 'subsample'}:
    print('Note: orchestration full-vs-subsample comparison needs both saved runs.')
    print("Run this notebook once with EVAL_SCOPE='full_corpus' and once with EVAL_SCOPE='subsample' to populate both rows.")

baseline_full_report_df.to_csv(REPORT_RESULTS_DIR / 'baseline_full_report_structure.csv', index=False)
orchestration_report_df.to_csv(REPORT_RESULTS_DIR / f'orchestration_{EVAL_SCOPE}_report_structure.csv', index=False)
full_system_report_df.to_csv(REPORT_RESULTS_DIR / f'baseline_plus_orchestration_{EVAL_SCOPE}.csv', index=False)
baseline_scope_report_df.to_csv(REPORT_RESULTS_DIR / 'baseline_subsample_vs_full_report_structure.csv', index=False)
orchestration_scope_report_df.to_csv(REPORT_RESULTS_DIR / 'orchestration_scope_comparison_report_structure.csv', index=False)

print(f'\nSaved report-aligned tables to {REPORT_RESULTS_DIR.resolve()}')

### Step 9 — Completion checklist before sharing

- [ ] Notebook runs top-to-bottom without errors on `EVAL_SCOPE = 'subsample'` first, then on `'full_corpus'`.
- [ ] `ConfidenceOrchestrator` returns `(answer, docs, trace)` reliably for all 6 query types.
- [ ] Trace dict shows different weight presets actually firing across query types.
- [ ] Quantitative metrics table saved (CSV + JSON) under a clear filename like `results/strategy_A_confidence_full.csv`.
- [ ] Efficiency metrics included (latency stats + retry count + gating count).
- [ ] Qualitative findings documented (explainability traces, complementarity table, failure categories).
- [ ] Comparison vs Voting baseline plotted (MRR bar + latency box).
- [ ] Optional weight-preset ablation included.
- [ ] Plots generated, labelled, and readable at report size.
- [ ] Artifacts exported (CSV/JSON) for reproducibility.
- [ ] Notes added in a final cell for what to merge into the team's combined notebook (which functions are reusable, which are A-specific).

This cell helps collaboration and keeps the branch handoff to Julia / the report stage clean.